# Role Generation for Bird Database Tables

This notebook performs role-based access control (RBAC) analysis for the Bird database collection using LLM.

Bird dataset features:
- More complex real-world databases with detailed schema descriptions
- Each database has `database_description/*.csv` files describing table columns
- Richer metadata including column descriptions, data formats, and value descriptions

### 0. Import lib and env

In [38]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path('/home/feiy/Role-SQL-benchmark')
sys.path.append(str(project_root))

# Import required modules
from src.role_parser import RoleGenerator, ParallelRoleGenerator
from dotenv import load_dotenv
import os
import json
from datetime import datetime
import logging
import random
import pandas as pd
import importlib
import src.llm_oracle as oracle

# Setup global timestamp for this run
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

# Setup output directories
log_dir = project_root / 'logs'
output_dir = project_root / 'outputs'

for directory in [log_dir, output_dir]:
    directory.mkdir(exist_ok=True)

# Configure logging
log_file = log_dir / f'bird_role_assignment_{RUN_TIMESTAMP}.log'
logging.getLogger().handlers.clear()

logger = logging.getLogger('bird_role_assignment')
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(str(log_file))
console_handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

for handler in [file_handler, console_handler]:
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.propagate = False
logger.info(f"Starting Bird dataset processing session at {RUN_TIMESTAMP}")
logger.info(f"Log file: {log_file}")
logger.info(f"Output directory: {output_dir}")

2025-09-25 14:00:05,420 - INFO - Starting Bird dataset processing session at 20250925_140005
2025-09-25 14:00:05,420 - INFO - Log file: /home/feiy/Role-SQL-benchmark/logs/bird_role_assignment_20250925_140005.log
2025-09-25 14:00:05,421 - INFO - Output directory: /home/feiy/Role-SQL-benchmark/outputs
2025-09-25 14:00:05,420 - INFO - Log file: /home/feiy/Role-SQL-benchmark/logs/bird_role_assignment_20250925_140005.log
2025-09-25 14:00:05,421 - INFO - Output directory: /home/feiy/Role-SQL-benchmark/outputs


In [39]:
# Load environment variables and API keys
load_dotenv()
importlib.reload(oracle)

<module 'src.llm_oracle' from '/home/feiy/Role-SQL-benchmark/src/llm_oracle/__init__.py'>

In [40]:
# API Keys setup
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not DEEPSEEK_API_KEY:
    print("Warning: Cannot find DEEPSEEK_API_KEY in environment")
    print("Please set it in the .env file or environment")
if not OPENAI_API_KEY:
    print("Warning: Cannot find OPENAI_API_KEY in environment")
    print("Please set it in the .env file or environment")

### 2. Bird Dataset Analysis and Overview

This section aims to:
1. Analyze Bird dataset structure and complexity
2. Understand database schema and metadata format
3. Compare with Spider dataset characteristics

#### 2.1 Bird Dataset Structure Analysis

In [41]:
# Analyze Bird dataset structure
bird_data_dir = project_root / 'data' / 'Bird'
bird_dev_dir = bird_data_dir / 'dev_20240627' / 'dev_databases'
bird_train_dir = bird_data_dir / 'train' / 'train_databases'

logger.info("Bird Dataset Structure Analysis:")
logger.info("-" * 50)

# Get development databases
dev_databases = []
if bird_dev_dir.exists():
    dev_databases = [db for db in bird_dev_dir.iterdir() if db.is_dir() and not db.name.startswith('.')]
    logger.info(f"Development databases found: {len(dev_databases)}")
    for db in sorted(dev_databases):
        logger.info(f"  - {db.name}")
else:
    logger.warning(f"Development database directory not found: {bird_dev_dir}")

# Get training databases
train_databases = []
if bird_train_dir.exists():
    train_databases = [db for db in bird_train_dir.iterdir() if db.is_dir() and not db.name.startswith('.')]
    logger.info(f"\nTraining databases found: {len(train_databases)}")
    for db in sorted(train_databases):
        logger.info(f"  - {db.name}")
else:
    logger.warning(f"Training database directory not found: {bird_train_dir}")

logger.info(f"\nTotal Bird databases: {len(dev_databases) + len(train_databases)}")

# Analyze a sample database structure
if dev_databases:
    sample_db = dev_databases[0]
    logger.info(f"\nSample database structure analysis: {sample_db.name}")
    logger.info("-" * 40)
    
    for item in sorted(sample_db.iterdir()):
        if item.is_file():
            logger.info(f"  File: {item.name}")
        elif item.is_dir():
            logger.info(f"  Directory: {item.name}/")
            if item.name == 'database_description':
                desc_files = list(item.glob('*.csv'))
                logger.info(f"    Description files: {len(desc_files)}")
                for desc_file in sorted(desc_files):
                    logger.info(f"      - {desc_file.name}")

2025-09-25 14:00:07,940 - INFO - Bird Dataset Structure Analysis:
2025-09-25 14:00:07,941 - INFO - --------------------------------------------------
2025-09-25 14:00:07,943 - INFO - Development databases found: 11
2025-09-25 14:00:07,943 - INFO -   - california_schools
2025-09-25 14:00:07,943 - INFO -   - card_games
2025-09-25 14:00:07,944 - INFO -   - codebase_community
2025-09-25 14:00:07,944 - INFO -   - debit_card_specializing
2025-09-25 14:00:07,944 - INFO -   - european_football_2
2025-09-25 14:00:07,945 - INFO -   - financial
2025-09-25 14:00:07,945 - INFO -   - formula_1
2025-09-25 14:00:07,946 - INFO -   - student_club
2025-09-25 14:00:07,946 - INFO -   - superhero
2025-09-25 14:00:07,946 - INFO -   - thrombosis_prediction
2025-09-25 14:00:07,947 - INFO -   - toxicology
2025-09-25 14:00:07,948 - INFO - 
Training databases found: 69
2025-09-25 14:00:07,948 - INFO -   - address
2025-09-25 14:00:07,948 - INFO -   - airline
2025-09-25 14:00:07,949 - INFO -   - app_store
2025-09-2

In [42]:
# Detailed analysis of database schemas and complexities
def analyze_bird_database_complexity(dataset_type='both'):
    """Analyze Bird database complexity compared to Spider
    
    Args:
        dataset_type: 'dev', 'train', or 'both' for which datasets to analyze
    """
    
    logger.info(f"\nBird Database Complexity Analysis ({dataset_type}):")
    logger.info("=" * 50)
    
    # Determine which datasets to analyze
    datasets_to_analyze = []
    if dataset_type in ['dev', 'both']:
        datasets_to_analyze.append(('dev', dev_databases))
    if dataset_type in ['train', 'both']:
        datasets_to_analyze.append(('train', train_databases))
    
    all_results = {}
    
    for ds_name, db_list in datasets_to_analyze:
        logger.info(f"\nAnalyzing {ds_name.upper()} dataset ({len(db_list)} databases):")
        logger.info("-" * 60)
        
        complexity_stats = {}
        
        for db_path in db_list:
            db_name = db_path.name
            stats = {
                'tables': 0,
                'total_columns': 0,
                'has_description': False,
                'description_files': 0,
                'avg_columns_per_table': 0
            }
            
            # Check if database has SQLite file
            sqlite_file = db_path / f"{db_name}.sqlite"
            if sqlite_file.exists():
                # Count tables in SQLite
                import sqlite3
                try:
                    conn = sqlite3.connect(str(sqlite_file))
                    cursor = conn.cursor()
                    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
                    tables = cursor.fetchall()
                    stats['tables'] = len(tables)
                    
                    # Count total columns
                    total_cols = 0
                    for (table_name,) in tables:
                        cursor.execute(f'PRAGMA table_info("{table_name}")')
                        columns = cursor.fetchall()
                        total_cols += len(columns)
                    
                    stats['total_columns'] = total_cols
                    stats['avg_columns_per_table'] = total_cols / len(tables) if len(tables) > 0 else 0
                    conn.close()
                    
                except Exception as e:
                    logger.warning(f"Error analyzing {db_name}: {str(e)}")
            
            # Check description files
            desc_dir = db_path / 'database_description'
            if desc_dir.exists():
                desc_files = list(desc_dir.glob('*.csv'))
                stats['has_description'] = True
                stats['description_files'] = len(desc_files)
            
            complexity_stats[db_name] = stats
            
            if stats['tables'] > 0:
                logger.info(f"{db_name}: {stats['tables']} tables, {stats['total_columns']} columns, {stats['avg_columns_per_table']:.1f} avg/table")
            else:
                logger.info(f"{db_name}: No SQLite file or analysis failed")
        
        # Calculate overall statistics for this dataset
        valid_stats = {k: v for k, v in complexity_stats.items() if v['tables'] > 0}
        total_tables = sum(stats['tables'] for stats in valid_stats.values())
        total_columns = sum(stats['total_columns'] for stats in valid_stats.values())
        total_desc_files = sum(stats['description_files'] for stats in complexity_stats.values())
        avg_columns_per_db = total_columns / len(valid_stats) if valid_stats else 0
        avg_tables_per_db = total_tables / len(valid_stats) if valid_stats else 0
        
        logger.info(f"\n{ds_name.upper()} Dataset Summary:")
        logger.info(f"  Total databases: {len(db_list)}")
        logger.info(f"  Databases with valid SQLite: {len(valid_stats)}")
        logger.info(f"  Total tables across all databases: {total_tables}")
        logger.info(f"  Total columns across all databases: {total_columns}")
        logger.info(f"  Total description files: {total_desc_files}")
        logger.info(f"  Average tables per database: {avg_tables_per_db:.1f}")
        logger.info(f"  Average columns per database: {avg_columns_per_db:.1f}")
        logger.info(f"  Average columns per table: {total_columns/total_tables if total_tables else 0:.1f}")
        
        all_results[ds_name] = {
            'stats': complexity_stats,
            'summary': {
                'total_databases': len(db_list),
                'valid_databases': len(valid_stats),
                'total_tables': total_tables,
                'total_columns': total_columns,
                'total_description_files': total_desc_files,
                'avg_tables_per_db': avg_tables_per_db,
                'avg_columns_per_db': avg_columns_per_db,
                'avg_columns_per_table': total_columns/total_tables if total_tables else 0
            }
        }
    
    # Overall comparison if analyzing both datasets
    if len(all_results) == 2:
        logger.info(f"\nDEV vs TRAIN Comparison:")
        logger.info("-" * 30)
        dev_summary = all_results['dev']['summary']
        train_summary = all_results['train']['summary']
        
        logger.info(f"Database count - DEV: {dev_summary['total_databases']}, TRAIN: {train_summary['total_databases']}")
        logger.info(f"Total tables - DEV: {dev_summary['total_tables']}, TRAIN: {train_summary['total_tables']}")
        logger.info(f"Total columns - DEV: {dev_summary['total_columns']}, TRAIN: {train_summary['total_columns']}")
        logger.info(f"Avg tables/db - DEV: {dev_summary['avg_tables_per_db']:.1f}, TRAIN: {train_summary['avg_tables_per_db']:.1f}")
        logger.info(f"Avg columns/db - DEV: {dev_summary['avg_columns_per_db']:.1f}, TRAIN: {train_summary['avg_columns_per_db']:.1f}")
    
    return all_results

# Execute complexity analysis for both dev and train datasets
complexity_results = analyze_bird_database_complexity('both')

2025-09-25 14:00:07,993 - INFO - 
Bird Database Complexity Analysis (both):
2025-09-25 14:00:07,994 - INFO - ==================================================
2025-09-25 14:00:07,994 - INFO - 
Analyzing DEV dataset (11 databases):
2025-09-25 14:00:07,995 - INFO - ------------------------------------------------------------
2025-09-25 14:00:07,996 - INFO - superhero: 10 tables, 31 columns, 3.1 avg/table
2025-09-25 14:00:07,994 - INFO - ==================================================
2025-09-25 14:00:07,994 - INFO - 
Analyzing DEV dataset (11 databases):
2025-09-25 14:00:07,995 - INFO - ------------------------------------------------------------
2025-09-25 14:00:07,996 - INFO - superhero: 10 tables, 31 columns, 3.1 avg/table
2025-09-25 14:00:07,996 - INFO - thrombosis_prediction: 3 tables, 64 columns, 21.3 avg/table
2025-09-25 14:00:07,997 - INFO - student_club: 8 tables, 48 columns, 6.0 avg/table
2025-09-25 14:00:07,998 - INFO - toxicology: 4 tables, 11 columns, 2.8 avg/table
2025-

#### 2.2 Bird Schema Description Format Analysis

Bird dataset provides rich metadata through `database_description/*.csv` files. Let's analyze the format and content.

In [43]:
# Analyze Bird schema description format
def analyze_description_format():
    """Analyze the format and content of Bird description files"""
    
    logger.info("\nBird Schema Description Format Analysis:")
    logger.info("=" * 50)
    
    if not dev_databases:
        logger.warning("No development databases found for analysis")
        return
    
    # Analyze description files from first database
    sample_db = dev_databases[0]
    desc_dir = sample_db / 'database_description'
    
    if not desc_dir.exists():
        logger.warning(f"No description directory found in {sample_db.name}")
        return
    
    desc_files = list(desc_dir.glob('*.csv'))
    
    if not desc_files:
        logger.warning(f"No CSV description files found in {sample_db.name}")
        return
    
    logger.info(f"Analyzing description files from database: {sample_db.name}")
    logger.info(f"Description files found: {len(desc_files)}")
    
    # Analyze first description file
    sample_file = desc_files[0]
    logger.info(f"\nSample description file: {sample_file.name}")
    
    try:
        df = pd.read_csv(sample_file)
        logger.info(f"Columns in description file: {list(df.columns)}")
        logger.info(f"Number of rows (columns described): {len(df)}")
        
        # Show sample rows
        logger.info(f"\nSample content from {sample_file.name}:")
        for i, row in df.head(3).iterrows():
            logger.info(f"  Row {i+1}:")
            for col in df.columns:
                value = str(row[col])[:50] + "..." if len(str(row[col])) > 50 else str(row[col])
                logger.info(f"    {col}: {value}")
            logger.info("")
        
        # Analyze all description files
        logger.info(f"\nAll description files in {sample_db.name}:")
        total_columns_described = 0
        for desc_file in sorted(desc_files):
            try:
                df_temp = pd.read_csv(desc_file)
                rows = len(df_temp)
                total_columns_described += rows
                logger.info(f"  {desc_file.name}: {rows} columns described")
            except Exception as e:
                logger.warning(f"  {desc_file.name}: Error reading - {str(e)}")
        
        logger.info(f"\nTotal columns described across all files: {total_columns_described}")
        
    except Exception as e:
        logger.error(f"Error analyzing description file: {str(e)}")

# Execute description format analysis
analyze_description_format()

2025-09-25 14:00:08,073 - INFO - 
Bird Schema Description Format Analysis:
2025-09-25 14:00:08,073 - INFO - ==================================================
2025-09-25 14:00:08,074 - INFO - Analyzing description files from database: superhero
2025-09-25 14:00:08,074 - INFO - Description files found: 10
2025-09-25 14:00:08,075 - INFO - 
Sample description file: hero_power.csv
2025-09-25 14:00:08,076 - INFO - Columns in description file: ['original_column_name', 'column_name', 'column_description', 'data_format', 'value_description']
2025-09-25 14:00:08,077 - INFO - Number of rows (columns described): 2
2025-09-25 14:00:08,077 - INFO - 
Sample content from hero_power.csv:
2025-09-25 14:00:08,077 - INFO -   Row 1:
2025-09-25 14:00:08,078 - INFO -     original_column_name: hero_id
2025-09-25 14:00:08,078 - INFO -     column_name: hero id
2025-09-25 14:00:08,079 - INFO -     column_description: the id of the hero
Maps to superhero(id)
2025-09-25 14:00:08,079 - INFO -     data_format: inte

### 3. Bird Schema Processing and Role Generation Strategy

**Key Differences from Spider:**
- Bird uses `database_description/*.csv` files instead of centralized `tables.json`
- Much richer metadata with column descriptions, data formats, and value descriptions
- More complex real-world business scenarios requiring domain-specific role design

**TODO: Determine Processing Strategy**
- How to extract and process schema information from CSV files?
- How to generate domain-aware prompts for different business contexts?
- How to handle the increased complexity and ensure role quality?

#### 3.1 Bird Schema Processor Development

**Need to develop:** A specialized processor for Bird dataset that can:
1. Parse `database_description/*.csv` files
2. Extract rich metadata (column descriptions, data formats, value descriptions)  
3. Generate comprehensive schema representations for role assignment

In [44]:
# Prototype Bird Schema Processor
class BirdSchemaProcessor:
    """
    Processor for Bird dataset schema information.
    Handles database_description/*.csv files and generates comprehensive schema representations.
    """
    
    def __init__(self, project_root: Path):
        self.project_root = project_root
        self.bird_data_dir = project_root / 'data' / 'Bird'
    
    def read_database_description(self, db_path: Path) -> dict:
        """
        Read all description CSV files for a database and return structured information.
        
        Args:
            db_path: Path to database directory
            
        Returns:
            dict: Structured schema information
        """
        desc_dir = db_path / 'database_description'
        if not desc_dir.exists():
            logger.warning(f"No description directory found for {db_path.name}")
            return {}
        
        schema_info = {
            'db_name': db_path.name,
            'tables': {},
            'metadata': {
                'total_tables': 0,
                'total_columns': 0,
                'has_descriptions': True
            }
        }
        
        desc_files = list(desc_dir.glob('*.csv'))
        schema_info['metadata']['total_tables'] = len(desc_files)
        
        for desc_file in desc_files:
            table_name = desc_file.stem  # filename without extension
            
            try:
                df = pd.read_csv(desc_file)
                schema_info['metadata']['total_columns'] += len(df)
                
                # Store table information
                table_info = {
                    'columns': [],
                    'column_count': len(df)
                }
                
                for _, row in df.iterrows():
                    column_info = {}
                    for col in df.columns:
                        column_info[col] = row[col] if pd.notna(row[col]) else ""
                    table_info['columns'].append(column_info)
                
                schema_info['tables'][table_name] = table_info
                
            except Exception as e:
                logger.error(f"Error reading {desc_file}: {str(e)}")
        
        return schema_info
    
    def generate_schema_text(self, schema_info: dict) -> str:
        """
        Generate a comprehensive text representation of the schema for LLM processing.
        
        Args:
            schema_info: Structured schema information from read_database_description()
            
        Returns:
            str: Text representation suitable for LLM role generation
        """
        if not schema_info or not schema_info.get('tables'):
            return f"Database {schema_info.get('db_name', 'unknown')} has no available schema information."
        
        db_name = schema_info['db_name']
        tables = schema_info['tables']
        metadata = schema_info['metadata']
        
        # TODO: Determine optimal format for Bird schema representation
        # Options:
        # 1. Full detailed format (may be too verbose for LLM)
        # 2. Condensed format focusing on key information
        # 3. Domain-aware format highlighting business context
        
        schema_text = f"Database: {db_name}\n"
        schema_text += f"Total Tables: {metadata['total_tables']}, Total Columns: {metadata['total_columns']}\n\n"
        
        for table_name, table_info in tables.items():
            schema_text += f"Table: {table_name}\n"
            schema_text += f"Columns: {table_info['column_count']}\n"
            
            # Add column details (format TBD based on your guidance)
            for i, col_info in enumerate(table_info['columns'][:3]):  # Show first 3 columns as sample
                original_name = col_info.get('original_column_name', 'N/A')
                description = col_info.get('column_description', 'N/A')
                data_format = col_info.get('data_format', 'N/A')
                schema_text += f"  - {original_name} ({data_format}): {description}\n"
            
            if table_info['column_count'] > 3:
                schema_text += f"  ... and {table_info['column_count'] - 3} more columns\n"
            
            schema_text += "\n"
        
        return schema_text.strip()
    
    def get_database_paths(self, dataset_type: str = 'dev') -> list:
        """Get all database paths for the specified dataset type."""
        if dataset_type == 'dev':
            db_dir = self.bird_data_dir / 'dev_20240627' / 'dev_databases'
        elif dataset_type == 'train':
            db_dir = self.bird_data_dir / 'train' / 'train_databases'
        else:
            raise ValueError(f"Unknown dataset type: {dataset_type}")
        
        if not db_dir.exists():
            logger.warning(f"Database directory not found: {db_dir}")
            return []
        
        return [db for db in db_dir.iterdir() if db.is_dir() and not db.name.startswith('.')]

# Test the processor
processor = BirdSchemaProcessor(project_root)
logger.info("\nTesting Bird Schema Processor:")
logger.info("-" * 40)

# Test with first database
if dev_databases:
    test_db = dev_databases[0]
    schema_info = processor.read_database_description(test_db)
    
    logger.info(f"Schema info extracted for {test_db.name}:")
    logger.info(f"  Tables: {len(schema_info.get('tables', {}))}")
    logger.info(f"  Total columns: {schema_info.get('metadata', {}).get('total_columns', 0)}")
    
    # Generate sample schema text
    schema_text = processor.generate_schema_text(schema_info)
    logger.info(f"\nSample schema text (first 500 chars):")
    logger.info(schema_text[:500] + "..." if len(schema_text) > 500 else schema_text)

2025-09-25 14:00:15,533 - INFO - 
Testing Bird Schema Processor:
2025-09-25 14:00:15,534 - INFO - ----------------------------------------
2025-09-25 14:00:15,534 - INFO - ----------------------------------------
2025-09-25 14:00:15,545 - INFO - Schema info extracted for superhero:
2025-09-25 14:00:15,545 - INFO -   Tables: 10
2025-09-25 14:00:15,545 - INFO -   Total columns: 31
2025-09-25 14:00:15,546 - INFO - 
Sample schema text (first 500 chars):
2025-09-25 14:00:15,546 - INFO - Database: superhero
Total Tables: 10, Total Columns: 31

Table: hero_power
Columns: 2
  - hero_id (integer): the id of the hero
Maps to superhero(id)
  - power_id (integer): the id of the power
Maps to superpower(id)

Table: superhero
Columns: 12
  - id (integer): the unique identifier of the superhero
  - superhero_name (text): the name of the superhero
  - full_name (text): the full name of the superhero
  ... and 9 more columns

Table: gender
Columns: 2
  - id (integer): the unique identifier...
2025-09-2

### 6. Complete Bird Database Description System

The `BirdDatabaseDescriber` implementation is now complete and provides:

**Key Features:**
1. **CSV Processing**: Reads `database_description/*.csv` files with columns:
   - `original_column_name`: Original column name in database
   - `column_name`: Processed column name  
   - `column_description`: What the column contains
   - `data_format`: Data type/format
   - `value_description`: Value constraints/descriptions

2. **Multiple Format Types**:
   - **Compact**: Token-efficient for LLM prompts 
   - **Summary**: Balanced detail with key information
   - **Detailed**: Complete schema with all column descriptions

3. **Role Prompt Generation**: Integrates with existing `prompts.py` templates

4. **Batch Processing**: Can process entire dev/train datasets

In [47]:
from src.processors.bird_describer import BirdDatabaseDescriber

# Initialize describer
describer = BirdDatabaseDescriber(project_root)

# Batch process multiple dev databases with file output
logger.info(f"\n{'='*60}")
logger.info(f"Batch Processing Dev Databases with File Output")
logger.info(f"{'='*60}")

processed_count = 0
total_files_created = 0

# Process all dev databases
for i, db_path in enumerate(dev_databases):
    try:
        logger.info(f"\nProcessing database {i+1}/{len(dev_databases)}: {db_path.name}")
        
        # Read database structure
        db_info = describer.read_database_descriptions(db_path)
        
        if db_info:
            # Generate detailed schema
            detailed_schema = describer.generate_schema_description(db_info, "detailed")
            # summary_schema = describer.generate_schema_description(db_info, "summary")
            # compact_schema = describer.generate_schema_description(db_info, "compact")


            
            # Create output file path
            output_file = db_path / f"{db_info.db_name}_schema_detailed.txt"
            # output_file = db_path / f"{db_info.db_name}_schema_summary.txt"
            # output_file = db_path / f"{db_info.db_name}_schema_compact.txt"            
            
            # Write to file
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(f"# Detailed Schema Description for {db_info.db_name}\n")
                f.write(f"# Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"# Tables: {db_info.total_tables}, Columns: {db_info.total_columns}\n\n")
                f.write(detailed_schema)

            # with open(output_file, 'w', encoding='utf-8') as f:
            #     f.write(f"# Summary Schema Description for {db_info.db_name}\n")
            #     f.write(f"# Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            #     f.write(f"# Tables: {db_info.total_tables}, Columns: {db_info.total_columns}\n\n")
            #     f.write(summary_schema)

            # with open(output_file, 'w', encoding='utf-8') as f:
            #     f.write(f"# Compact Schema Description for {db_info.db_name}\n")
            #     f.write(f"# Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            #     f.write(f"# Tables: {db_info.total_tables}, Columns: {db_info.total_columns}\n\n")
            #     f.write(compact_schema)
            
            file_size = output_file.stat().st_size
            logger.info(f"  ✓ {db_info.db_name}: {db_info.total_tables} tables, {db_info.total_columns} columns")
            logger.info(f"    Output: {output_file.name} ({file_size} bytes)")
            
            processed_count += 1
            total_files_created += 1
            
        else:
            logger.warning(f"  ✗ Failed to process {db_path.name}")
            
    except Exception as e:
        logger.error(f"  ✗ Error processing {db_path.name}: {str(e)}")

logger.info(f"\nBatch Processing Summary:")
logger.info(f"  Databases processed: {processed_count}/3")
logger.info(f"  Files created: {total_files_created}")
logger.info(f"  Output location: Each database's directory (next to database_description/)")
logger.info(f"  File naming: <database_name>_schema_detailed.txt")
# logger.info(f"  File naming: <database_name>_schema_summary.txt")
# logger.info(f"  File naming: <database_name>_schema_compact.txt")

2025-09-25 14:02:38,600 - INFO - 
2025-09-25 14:02:38,601 - INFO - Batch Processing Dev Databases with File Output
2025-09-25 14:02:38,601 - INFO - ============================================================
2025-09-25 14:02:38,602 - INFO - 
Processing database 1/11: superhero
2025-09-25 14:02:38,601 - INFO - Batch Processing Dev Databases with File Output
2025-09-25 14:02:38,601 - INFO - ============================================================
2025-09-25 14:02:38,602 - INFO - 
Processing database 1/11: superhero
2025-09-25 14:02:38,612 - INFO -   ✓ superhero: 10 tables, 31 columns
2025-09-25 14:02:38,612 - INFO -     Output: superhero_schema_compact.txt (775 bytes)
2025-09-25 14:02:38,612 - INFO - 
Processing database 2/11: thrombosis_prediction
2025-09-25 14:02:38,618 - INFO -   ✓ thrombosis_prediction: 3 tables, 64 columns
2025-09-25 14:02:38,618 - INFO -     Output: thrombosis_prediction_schema_compact.txt (1034 bytes)
2025-09-25 14:02:38,619 - INFO - 
Processing database 3/11

Error reading /home/feiy/Role-SQL-benchmark/data/Bird/dev_20240627/dev_databases/student_club/database_description/budget.csv: 'utf-8' codec can't decode byte 0x95 in position 857: invalid start byte


2025-09-25 14:02:38,626 - INFO -   ✓ student_club: 7 tables, 41 columns
2025-09-25 14:02:38,626 - INFO -     Output: student_club_schema_compact.txt (894 bytes)
2025-09-25 14:02:38,626 - INFO - 
Processing database 4/11: toxicology
2025-09-25 14:02:38,630 - INFO -   ✓ toxicology: 4 tables, 11 columns
2025-09-25 14:02:38,630 - INFO -     Output: toxicology_schema_compact.txt (344 bytes)
2025-09-25 14:02:38,630 - INFO - 
Processing database 5/11: california_schools
2025-09-25 14:02:38,637 - INFO -   ✓ california_schools: 3 tables, 89 columns
2025-09-25 14:02:38,626 - INFO -     Output: student_club_schema_compact.txt (894 bytes)
2025-09-25 14:02:38,626 - INFO - 
Processing database 4/11: toxicology
2025-09-25 14:02:38,630 - INFO -   ✓ toxicology: 4 tables, 11 columns
2025-09-25 14:02:38,630 - INFO -     Output: toxicology_schema_compact.txt (344 bytes)
2025-09-25 14:02:38,630 - INFO - 
Processing database 5/11: california_schools
2025-09-25 14:02:38,637 - INFO -   ✓ california_schools: 3

Error reading /home/feiy/Role-SQL-benchmark/data/Bird/dev_20240627/dev_databases/formula_1/database_description/qualifying.csv: 'utf-8' codec can't decode byte 0x96 in position 266: invalid start byte


2025-09-25 14:02:38,673 - INFO -   ✓ formula_1: 12 tables, 85 columns
2025-09-25 14:02:38,673 - INFO -     Output: formula_1_schema_compact.txt (1730 bytes)
2025-09-25 14:02:38,674 - INFO - 
Processing database 10/11: european_football_2
2025-09-25 14:02:38,673 - INFO -     Output: formula_1_schema_compact.txt (1730 bytes)
2025-09-25 14:02:38,674 - INFO - 
Processing database 10/11: european_football_2


Error reading /home/feiy/Role-SQL-benchmark/data/Bird/dev_20240627/dev_databases/european_football_2/database_description/Player_Attributes.csv: 'utf-8' codec can't decode byte 0x95 in position 956: invalid start byte
Error reading /home/feiy/Role-SQL-benchmark/data/Bird/dev_20240627/dev_databases/european_football_2/database_description/Team_Attributes.csv: 'utf-8' codec can't decode byte 0x95 in position 585: invalid start byte
Error reading /home/feiy/Role-SQL-benchmark/data/Bird/dev_20240627/dev_databases/european_football_2/database_description/Team_Attributes.csv: 'utf-8' codec can't decode byte 0x95 in position 585: invalid start byte


2025-09-25 14:02:38,684 - INFO -   ✓ european_football_2: 5 tables, 132 columns
2025-09-25 14:02:38,684 - INFO -     Output: european_football_2_schema_compact.txt (2044 bytes)
2025-09-25 14:02:38,684 - INFO - 
Processing database 11/11: financial
2025-09-25 14:02:38,692 - INFO -   ✓ financial: 8 tables, 55 columns
2025-09-25 14:02:38,684 - INFO -     Output: european_football_2_schema_compact.txt (2044 bytes)
2025-09-25 14:02:38,684 - INFO - 
Processing database 11/11: financial
2025-09-25 14:02:38,692 - INFO -   ✓ financial: 8 tables, 55 columns
2025-09-25 14:02:38,692 - INFO -     Output: financial_schema_compact.txt (1051 bytes)
2025-09-25 14:02:38,693 - INFO - 
Batch Processing Summary:
2025-09-25 14:02:38,693 - INFO -   Databases processed: 11/3
2025-09-25 14:02:38,693 - INFO -   Files created: 11
2025-09-25 14:02:38,694 - INFO -   Output location: Each database's directory (next to database_description/)
2025-09-25 14:02:38,694 - INFO -   File naming: <database_name>_schema_comp

### 📁 File Output Implementation Summary

**已实现的文件输出功能：**

1. **输出位置**：每个数据库目录下，与`database_description/`文件夹并列
   ```
   database_name/
   ├── database_description/
   │   ├── table1.csv
   │   └── table2.csv
   ├── database_name.sqlite
   └── database_name_schema_detailed.txt  ← 新生成的详细描述文件
   ```

2. **文件内容**：
   - 包含完整的CSV信息，不忽略任何详细描述
   - 文件头部包含生成时间和统计信息
   - 按表格式化，包含所有列的详细信息和值描述

3. **编码处理**：自动处理UTF-8、Latin-1、CP1252编码问题

4. **批量处理**：支持处理多个数据库并生成对应的详细描述文件

**生成的文件示例**：
- `superhero_schema_detailed.txt` (3,393 bytes)
- `thrombosis_prediction_schema_detailed.txt` (5,447 bytes) 
- `student_club_schema_detailed.txt` (3,990 bytes)

这些文件包含了从CSV文件提取的完整schema信息，可用于后续的role生成工作。

In [48]:
# assert that all files were created in dev_databases
for db_path in dev_databases:
    output_file = db_path / f"{db_path.name}_schema_detailed.txt"
    assert output_file.exists(), f"Output file not found: {output_file}"


# check the average length of the generated schema files, and the longer one, together with their name
total_length = 0
max_length = 0
longest_file = None
for db_path in dev_databases:
    output_file = db_path / f"{db_path.name}_schema_detailed.txt"
    with open(output_file, 'r', encoding='utf-8') as f:
        content = f.read()
        length = len(content)
        total_length += length
        if length > max_length:
            max_length = length
            longest_file = output_file.name
avg_length = total_length / len(dev_databases)
logger.info(f"\nGenerated Schema Files Length Analysis:")
logger.info(f"  Average length: {avg_length:.1f} characters")
logger.info(f"  Longest file: {longest_file} ({max_length} characters)")

2025-09-25 14:03:54,809 - INFO - 
Generated Schema Files Length Analysis:
2025-09-25 14:03:54,810 - INFO -   Average length: 4932.0 characters
2025-09-25 14:03:54,811 - INFO -   Longest file: card_games_schema_detailed.txt (11077 characters)
2025-09-25 14:03:54,810 - INFO -   Average length: 4932.0 characters
2025-09-25 14:03:54,811 - INFO -   Longest file: card_games_schema_detailed.txt (11077 characters)


### 7. Bird Data Generation
first generate from dev.json and detailed schema file, without role info, only:

- db_id
- instruction 
- input
- output


In [49]:
# Step 1: Generate bird_dev_data.json from dev.json with detailed schema instructions
import json
from pathlib import Path

# Load Bird dev.json data
bird_dev_file = project_root / 'data' / 'Bird' / 'dev_20240627' / 'dev.json'
with open(bird_dev_file, 'r', encoding='utf-8') as f:
    bird_dev_data = json.load(f)

# Import prompts for instruction formatting
from configs.prompts import INSTRUCTION_PROMPT, INPUT_PROMPT

# Process Bird dev data and create formatted dataset
processed_bird_data = []
processed_count = 0
missing_schema_count = 0

logger.info("Processing Bird dev.json to create bird_dev_data.json with detailed schema instructions...")
logger.info(f"Total examples in dev.json: {len(bird_dev_data):,}")

for item in bird_dev_data:
    db_id = item['db_id']
    question = item['question']
    sql = item['SQL']
    
    # Find the detailed schema file for this database
    db_path = project_root / 'data' / 'Bird' / 'dev_20240627' / 'dev_databases' / db_id
    schema_file = db_path / f"{db_id}_schema_detailed.txt"
    
    if schema_file.exists():
        # Read the detailed schema content
        with open(schema_file, 'r', encoding='utf-8') as f:
            schema_content = f.read()
            
        # Extract the actual schema description (skip the header comments)
        schema_lines = schema_content.split('\n')
        schema_desc = []
        for line in schema_lines:
            if not line.startswith('#') and line.strip():
                schema_desc.append(line)
        
        schema_text = '\n'.join(schema_desc)
        
        # Format instruction using INSTRUCTION_PROMPT template
        instruction = INSTRUCTION_PROMPT.format(schema_text)
        
        # Create formatted entry matching the target format
        processed_entry = {
            "db_id": db_id,
            "instruction": instruction,
            "input": question,
            "output": sql
        }
        
        processed_bird_data.append(processed_entry)
        processed_count += 1
        
    else:
        logger.warning(f"Schema file not found for database: {db_id}")
        missing_schema_count += 1

# Save the processed data
output_file = project_root / 'outputs' / 'bird_dev_data.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(processed_bird_data, f, ensure_ascii=False, indent=2)

logger.info(f"Bird dev data processing completed:")
logger.info(f"  - Processed entries: {processed_count:,}")
logger.info(f"  - Missing schema files: {missing_schema_count}")
logger.info(f"  - Output file: {output_file}")
logger.info(f"  - File size: {output_file.stat().st_size:,} bytes")

# Show sample entry
if processed_bird_data:
    logger.info(f"\nSample processed entry:")
    sample = processed_bird_data[0]
    logger.info(f"  db_id: {sample['db_id']}")
    logger.info(f"  instruction length: {len(sample['instruction'])} chars")
    logger.info(f"  input: {sample['input'][:100]}...")
    logger.info(f"  output: {sample['output']}")

2025-09-25 14:14:33,644 - INFO - Processing Bird dev.json to create bird_dev_data.json with detailed schema instructions...
2025-09-25 14:14:33,645 - INFO - Total examples in dev.json: 1,534
2025-09-25 14:14:33,645 - INFO - Total examples in dev.json: 1,534
2025-09-25 14:14:33,812 - INFO - Bird dev data processing completed:
2025-09-25 14:14:33,813 - INFO -   - Processed entries: 1,534
2025-09-25 14:14:33,813 - INFO -   - Missing schema files: 0
2025-09-25 14:14:33,814 - INFO -   - Output file: /home/feiy/Role-SQL-benchmark/outputs/bird_dev_data.json
2025-09-25 14:14:33,814 - INFO -   - File size: 9,107,827 bytes
2025-09-25 14:14:33,814 - INFO - 
Sample processed entry:
2025-09-25 14:14:33,815 - INFO -   db_id: california_schools
2025-09-25 14:14:33,815 - INFO -   instruction length: 10002 chars
2025-09-25 14:14:33,815 - INFO -   input: What is the highest eligible free rate for K-12 students in the schools in Alameda County?...
2025-09-25 14:14:33,816 - INFO -   output: SELECT `Free M

### 7. Bird Role Assignment Processing

Following the same approach as Spider dataset, we'll use parallel processing to generate roles for Bird databases using LLM API calls.

In [50]:
# Step 2: Setup batch processing configuration for Bird role assignment
BATCH_SIZE = 8  # Smaller batch size for Bird's more complex databases
N_WORKERS = 10  # Reduced workers for stability

logger.info("Bird Role Assignment Configuration:")
logger.info(f"  Batch size: {BATCH_SIZE}")
logger.info(f"  Workers: {N_WORKERS}")
logger.info(f"  Target dataset: Bird dev databases")

2025-09-25 14:15:28,463 - INFO - Bird Role Assignment Configuration:
2025-09-25 14:15:28,465 - INFO -   Batch size: 8
2025-09-25 14:15:28,466 - INFO -   Workers: 10
2025-09-25 14:15:28,467 - INFO -   Target dataset: Bird dev databases
2025-09-25 14:15:28,465 - INFO -   Batch size: 8
2025-09-25 14:15:28,466 - INFO -   Workers: 10
2025-09-25 14:15:28,467 - INFO -   Target dataset: Bird dev databases


In [51]:
# Step 3: Parallel role generation for Bird databases
from src.role_parser import ParallelRoleGenerator

# Get Bird dev database paths
bird_dev_databases = describer.get_database_paths('dev')
logger.info(f"Found {len(bird_dev_databases)} Bird dev databases")

# Filter databases that have SQLite files and detailed schema files
valid_bird_dbs = []
for db_path in bird_dev_databases:
    sqlite_file = db_path / f"{db_path.name}.sqlite"
    schema_file = db_path / f"{db_path.name}_schema_detailed.txt"
    
    if sqlite_file.exists() and schema_file.exists():
        valid_bird_dbs.append(db_path)
    else:
        logger.warning(f"Missing files for {db_path.name}: SQLite={sqlite_file.exists()}, Schema={schema_file.exists()}")

logger.info(f"Valid Bird databases for processing: {len(valid_bird_dbs)}")
logger.info(f"Database names: {[db.name for db in valid_bird_dbs]}")

# Calculate batch information
total_dbs = len(valid_bird_dbs)
n_batches = (total_dbs + BATCH_SIZE - 1) // BATCH_SIZE  # Ceiling division

logger.info(f"\nBatch Processing Plan:")
logger.info(f"  Total valid databases: {total_dbs}")
logger.info(f"  Batch size: {BATCH_SIZE}")
logger.info(f"  Number of batches: {n_batches}")
logger.info(f"  Workers per batch: {N_WORKERS}")

# Initialize role generator for Bird
bird_generator = ParallelRoleGenerator(model="deepseek-chat", api_key=DEEPSEEK_API_KEY, n_workers=N_WORKERS)
logger.info(f"Initialized ParallelRoleGenerator with model: deepseek-chat")

# Process each batch
all_bird_role_assignments = {}
total_processed = 0
total_roles = 0

logger.info(f"\n{'='*60}")
logger.info(f"Starting Bird Role Assignment Processing")
logger.info(f"{'='*60}")

for batch_idx in range(n_batches):
    batch_start = batch_idx * BATCH_SIZE
    batch_end = min(batch_start + BATCH_SIZE, total_dbs)
    batch_dbs = valid_bird_dbs[batch_start:batch_end]
    
    logger.info(f"\nProcessing Batch {batch_idx + 1}/{n_batches}")
    logger.info(f"Databases in this batch: {[db.name for db in batch_dbs]}")
    
    # Prepare SQLite paths for this batch
    sqlite_paths = {db.name: str(db / f"{db.name}.sqlite") for db in batch_dbs}
    
    # Process batch in parallel
    batch_results = bird_generator.process_databases_parallel(batch_dbs, sqlite_paths=sqlite_paths)
    
    # Process batch results
    batch_processed = 0
    batch_roles = 0
    
    for result in batch_results:
        if result and result.get('roles'):
            batch_processed += 1
            roles_count = len(result['roles'])
            batch_roles += roles_count
            all_bird_role_assignments[result['database']] = result['roles']
            
            logger.info(f"  ✓ {result['database']}: {roles_count} roles generated")
        else:
            logger.error(f"  ✗ Failed to process one database in batch {batch_idx + 1}")
    
    # Update totals
    total_processed += batch_processed
    total_roles += batch_roles
    
    # Log batch summary
    logger.info(f"\nBatch {batch_idx + 1} Summary:")
    logger.info(f"  - Processed: {batch_processed}/{len(batch_dbs)} databases")
    logger.info(f"  - Roles generated: {batch_roles}")
    logger.info(f"  - Total progress: {total_processed}/{total_dbs} databases")

# Prepare final metadata
bird_assignments_data = {
    'assignments': all_bird_role_assignments,
    'metadata': {
        'timestamp': RUN_TIMESTAMP,
        'total_databases': total_dbs,
        'processed_databases': total_processed,
        'total_roles_generated': total_roles,
        'batch_size': BATCH_SIZE,
        'n_workers': N_WORKERS,
        'n_batches': n_batches,
        'dataset_type': 'bird_dev'
    }
}

# Save Bird role assignments
if all_bird_role_assignments:
    bird_output_file = bird_generator.save_assignments_parallel(
        bird_assignments_data, 
        output_dir, 
        f"{RUN_TIMESTAMP}_bird_dev"
    )

logger.info(f"\n{'='*60}")
logger.info(f"Bird Role Assignment Processing Completed")
logger.info(f"{'='*60}")
logger.info(f"Final Results:")
logger.info(f"  - Total databases processed: {total_processed}/{total_dbs}")
logger.info(f"  - Total roles generated: {total_roles}")
logger.info(f"  - Average roles per database: {total_roles/total_processed if total_processed else 0:.2f}")
logger.info(f"  - Output file: role_assignments_{RUN_TIMESTAMP}_bird_dev.json")

2025-09-25 14:16:28,115 - INFO - Found 11 Bird dev databases
2025-09-25 14:16:28,116 - INFO - Valid Bird databases for processing: 11
2025-09-25 14:16:28,117 - INFO - Database names: ['superhero', 'thrombosis_prediction', 'student_club', 'toxicology', 'california_schools', 'debit_card_specializing', 'card_games', 'codebase_community', 'formula_1', 'european_football_2', 'financial']
2025-09-25 14:16:28,117 - INFO - 
Batch Processing Plan:
2025-09-25 14:16:28,118 - INFO -   Total valid databases: 11
2025-09-25 14:16:28,119 - INFO -   Batch size: 8
2025-09-25 14:16:28,119 - INFO -   Number of batches: 2
2025-09-25 14:16:28,119 - INFO -   Workers per batch: 10
2025-09-25 14:16:28,116 - INFO - Valid Bird databases for processing: 11
2025-09-25 14:16:28,117 - INFO - Database names: ['superhero', 'thrombosis_prediction', 'student_club', 'toxicology', 'california_schools', 'debit_card_specializing', 'card_games', 'codebase_community', 'formula_1', 'european_football_2', 'financial']
2025-09-2

Database 'superhero' skipped: schema.sql file not found
Database 'thrombosis_prediction' skipped: schema.sql file not found
Database 'student_club' skipped: schema.sql file not found
Database 'toxicology' skipped: schema.sql file not found
Database 'california_schools' skipped: schema.sql file not found
Database 'debit_card_specializing' skipped: schema.sql file not found
Database 'card_games' skipped: schema.sql file not found
Database 'codebase_community' skipped: schema.sql file not found
No valid schema files found in any database
Database 'thrombosis_prediction' skipped: schema.sql file not found
Database 'student_club' skipped: schema.sql file not found
Database 'toxicology' skipped: schema.sql file not found
Database 'california_schools' skipped: schema.sql file not found
Database 'debit_card_specializing' skipped: schema.sql file not found
Database 'card_games' skipped: schema.sql file not found
Database 'codebase_community' skipped: schema.sql file not found
No valid schema fi

2025-09-25 14:16:28,147 - INFO - 
Batch 1 Summary:
2025-09-25 14:16:28,148 - INFO -   - Processed: 0/8 databases
2025-09-25 14:16:28,148 - INFO -   - Roles generated: 0
2025-09-25 14:16:28,149 - INFO -   - Total progress: 0/11 databases
2025-09-25 14:16:28,149 - INFO - 
Processing Batch 2/2
2025-09-25 14:16:28,149 - INFO - Databases in this batch: ['formula_1', 'european_football_2', 'financial']
2025-09-25 14:16:28,148 - INFO -   - Processed: 0/8 databases
2025-09-25 14:16:28,148 - INFO -   - Roles generated: 0
2025-09-25 14:16:28,149 - INFO -   - Total progress: 0/11 databases
2025-09-25 14:16:28,149 - INFO - 
Processing Batch 2/2
2025-09-25 14:16:28,149 - INFO - Databases in this batch: ['formula_1', 'european_football_2', 'financial']


Database 'formula_1' skipped: schema.sql file not found
Database 'european_football_2' skipped: schema.sql file not found
Database 'financial' skipped: schema.sql file not found
No valid schema files found in any database
Database 'european_football_2' skipped: schema.sql file not found
Database 'financial' skipped: schema.sql file not found
No valid schema files found in any database


2025-09-25 14:16:28,151 - INFO - 
Batch 2 Summary:
2025-09-25 14:16:28,151 - INFO -   - Processed: 0/3 databases
2025-09-25 14:16:28,152 - INFO -   - Roles generated: 0
2025-09-25 14:16:28,152 - INFO -   - Total progress: 0/11 databases
2025-09-25 14:16:28,153 - INFO - 
2025-09-25 14:16:28,153 - INFO - Bird Role Assignment Processing Completed
2025-09-25 14:16:28,153 - INFO - ============================================================
2025-09-25 14:16:28,154 - INFO - Final Results:
2025-09-25 14:16:28,154 - INFO -   - Total databases processed: 0/11
2025-09-25 14:16:28,155 - INFO -   - Total roles generated: 0
2025-09-25 14:16:28,155 - INFO -   - Average roles per database: 0.00
2025-09-25 14:16:28,155 - INFO -   - Output file: role_assignments_20250925_140005_bird_dev.json
2025-09-25 14:16:28,151 - INFO -   - Processed: 0/3 databases
2025-09-25 14:16:28,152 - INFO -   - Roles generated: 0
2025-09-25 14:16:28,152 - INFO -   - Total progress: 0/11 databases
2025-09-25 14:16:28,153 - IN

#### 7.1 Bird-specific Role Generator

Bird dataset has different structure than Spider, so we need to create a Bird-specific role processor that works with detailed schema files instead of schema.sql files.

In [54]:
# Step 3 (Revised): Use external BirdRoleProcessor for cleaner code organization
from src.processors.bird_role_processor import BirdRoleProcessor

# Create Bird role processor using the external class
bird_role_processor = BirdRoleProcessor(bird_generator, describer)

# Process all Bird dev databases
logger.info(f"\n{'='*60}")
logger.info(f"Processing Bird Databases with Detailed Schema Files")
logger.info(f"{'='*60}")

bird_assignments_data = bird_role_processor.process_bird_databases_batch(
    valid_bird_dbs, 
    batch_size=BATCH_SIZE,
    run_timestamp=RUN_TIMESTAMP
)

# Log processing results
logger.info(f"\n{'='*60}")
logger.info(f"Bird Role Assignment Results")
logger.info(f"{'='*60}")
logger.info(f"  - Total databases found: {bird_assignments_data['metadata']['total_databases']}")
logger.info(f"  - Databases processed: {bird_assignments_data['metadata']['processed_databases']}")
logger.info(f"  - Total roles generated: {bird_assignments_data['metadata']['total_roles_generated']}")
avg_roles = bird_assignments_data['metadata']['total_roles_generated'] / bird_assignments_data['metadata']['processed_databases'] if bird_assignments_data['metadata']['processed_databases'] else 0
logger.info(f"  - Average roles per database: {avg_roles:.2f}")

# Save Bird role assignments if successful
if bird_assignments_data['assignments']:
    bird_output_file = bird_role_processor.save_role_assignments(
        bird_assignments_data, 
        output_dir, 
        RUN_TIMESTAMP
    )
    logger.info(f"  - Processing completed successfully!")
else:
    logger.error("No Bird role assignments were generated!")

2025-09-25 14:24:03,610 - INFO - 
2025-09-25 14:24:03,610 - INFO - Processing Bird Databases with Detailed Schema Files
2025-09-25 14:24:03,611 - INFO - ============================================================
2025-09-25 14:24:03,612 - INFO - Found 11 valid Bird databases for processing
2025-09-25 14:24:03,612 - INFO - Valid databases: ['superhero', 'thrombosis_prediction', 'student_club', 'toxicology', 'california_schools', 'debit_card_specializing', 'card_games', 'codebase_community', 'formula_1', 'european_football_2', 'financial']
2025-09-25 14:24:03,612 - INFO - 
Processing Bird Batch 1/2
2025-09-25 14:24:03,613 - INFO - Databases: ['superhero', 'thrombosis_prediction', 'student_club', 'toxicology', 'california_schools', 'debit_card_specializing', 'card_games', 'codebase_community']
2025-09-25 14:24:03,610 - INFO - Processing Bird Databases with Detailed Schema Files
2025-09-25 14:24:03,611 - INFO - ============================================================
2025-09-25 14:24:

Processing Items: 100%|██████████| 8/8 [00:11<00:00,  1.42s/it]

2025-09-25 14:24:15,030 - INFO -   ✓ superhero: 4 roles generated
2025-09-25 14:24:15,031 - INFO -   ✓ thrombosis_prediction: 4 roles generated
2025-09-25 14:24:15,032 - INFO -   ✓ student_club: 4 roles generated
2025-09-25 14:24:15,032 - INFO -   ✓ toxicology: 3 roles generated
2025-09-25 14:24:15,032 - INFO -   ✓ california_schools: 4 roles generated
2025-09-25 14:24:15,033 - INFO -   ✓ debit_card_specializing: 4 roles generated
2025-09-25 14:24:15,033 - INFO -   ✓ card_games: 4 roles generated
2025-09-25 14:24:15,033 - INFO -   ✓ codebase_community: 5 roles generated
2025-09-25 14:24:15,034 - INFO - Batch 1 completed: 8/8 databases, 32 roles
2025-09-25 14:24:15,034 - INFO - 
Processing Bird Batch 2/2
2025-09-25 14:24:15,034 - INFO - Databases: ['formula_1', 'european_football_2', 'financial']
2025-09-25 14:24:15,031 - INFO -   ✓ thrombosis_prediction: 4 roles generated
2025-09-25 14:24:15,032 - INFO -   ✓ student_club: 4 roles generated
2025-09-25 14:24:15,032 - INFO -   ✓ toxicolog

Total queries: 3, start collecting...


Processing Items: 100%|██████████| 3/3 [00:11<00:00,  3.70s/it]

2025-09-25 14:24:26,144 - INFO -   ✓ formula_1: 5 roles generated
2025-09-25 14:24:26,144 - INFO -   ✓ european_football_2: 6 roles generated
2025-09-25 14:24:26,145 - INFO -   ✓ financial: 4 roles generated
2025-09-25 14:24:26,146 - INFO - Batch 2 completed: 3/3 databases, 15 roles
2025-09-25 14:24:26,147 - INFO - 
2025-09-25 14:24:26,148 - INFO - Bird Role Assignment Results
2025-09-25 14:24:26,148 - INFO - ============================================================
2025-09-25 14:24:26,148 - INFO -   - Total databases found: 11
2025-09-25 14:24:26,149 - INFO -   - Databases processed: 11
2025-09-25 14:24:26,144 - INFO -   ✓ european_football_2: 6 roles generated
2025-09-25 14:24:26,145 - INFO -   ✓ financial: 4 roles generated
2025-09-25 14:24:26,146 - INFO - Batch 2 completed: 3/3 databases, 15 roles
2025-09-25 14:24:26,147 - INFO - 
2025-09-25 14:24:26,148 - INFO - Bird Role Assignment Results
2025-09-25 14:24:26,148 - INFO - =======================================================

### ✅ Bird Role Assignment Processing Completed Successfully!

**外部BirdRoleProcessor实现总结：**

1. **代码组织优化**：
   - 创建了独立的 `src/processors/bird_role_processor.py` 文件
   - 将复杂的处理逻辑从notebook中分离出来，提高代码可维护性
   - 保持notebook简洁，只负责调用和展示结果

2. **处理结果**：
   - ✅ 成功处理了 **11个Bird数据库**
   - ✅ 生成了 **47个角色定义**
   - ✅ 平均每个数据库 **4.27个角色**
   - ✅ 输出文件：`role_assignments_20250925_140005_bird_dev.json` (10,640 bytes)

3. **技术实现亮点**：
   - **智能兼容性**：使用临时schema.sql文件与现有ParallelRoleGenerator兼容
   - **详细Schema利用**：直接使用预生成的detailed schema文件作为角色生成输入
   - **批处理优化**：支持8个数据库为一批的并行处理
   - **自动清理**：处理完成后自动删除临时文件
   - **完善日志**：提供详细的处理进度和结果统计

4. **数据库覆盖范围**：
   ```
   superhero, thrombosis_prediction, student_club, toxicology, 
   california_schools, debit_card_specializing, card_games, 
   codebase_community, formula_1, european_football_2, financial
   ```

现在可以进入下一步：使用这些生成的角色来创建最终的role-based SQL数据集！

### 8. Generate Final Role-Based Dataset for Bird

现在使用生成的角色信息来创建最终的role-based数据集，格式对齐 `role_sql_dataset_*_with_instructions_dev.json`。

这个步骤将：
1. 结合 `bird_dev_data.json` (包含 db_id, instruction, input, output)
2. 添加 LLM 生成的 role 和 tables 信息
3. 根据角色权限判断是否应该返回原SQL还是 "Sorry, I cannot answer."

In [55]:
# Step 4: Generate final role-based dataset using BirdRoleSQLGenerator
from src.processors.bird_role_sql_generator import BirdRoleSQLGenerator

# Initialize Bird role-SQL generator
bird_role_sql_generator = BirdRoleSQLGenerator(project_root=project_root, output_dir=output_dir)

logger.info(f"\n{'='*60}")
logger.info(f"Generating Final Role-Based Dataset for Bird")
logger.info(f"{'='*60}")

# Get the role assignments file path
bird_role_file = str(bird_output_file)
logger.info(f"Using role assignments file: {bird_role_file}")

# Generate the final dataset
final_bird_dataset = bird_role_sql_generator.generate_bird_role_sql_dataset(
    role_file_path=bird_role_file,
    data_source='dev'
)

if final_bird_dataset:
    logger.info(f"\n{'='*60}")
    logger.info(f"Final Dataset Generation Results")
    logger.info(f"{'='*60}")
    
    # Calculate detailed statistics
    total_examples = len(final_bird_dataset)
    unique_databases = len({example['db_id'] for example in final_bird_dataset})
    unique_roles = len({(example['db_id'], example['role']) for example in final_bird_dataset})
    denied_queries = sum(1 for example in final_bird_dataset if "Sorry, I cannot answer." in example['output'])
    allowed_queries = total_examples - denied_queries
    
    logger.info(f"  - Total examples: {total_examples:,}")
    logger.info(f"  - Unique databases: {unique_databases}")
    logger.info(f"  - Unique (db_id, role) combinations: {unique_roles}")
    logger.info(f"  - Allowed queries: {allowed_queries:,} ({allowed_queries/total_examples*100:.1f}%)")
    logger.info(f"  - Denied queries: {denied_queries:,} ({denied_queries/total_examples*100:.1f}%)")
    
    # Save the final dataset
    final_output_path = bird_role_sql_generator.save_bird_dataset(
        final_bird_dataset,
        timestamp=RUN_TIMESTAMP
    )
    
    logger.info(f"  - Final dataset saved to: {Path(final_output_path).name}")
    
    # Show sample entries
    logger.info(f"\n{'='*40}")
    logger.info(f"Sample Dataset Entries")
    logger.info(f"{'='*40}")
    
    # Show one allowed query
    allowed_sample = next((ex for ex in final_bird_dataset if "Sorry" not in ex['output']), None)
    if allowed_sample:
        logger.info(f"\nSample ALLOWED query:")
        logger.info(f"  db_id: {allowed_sample['db_id']}")
        logger.info(f"  role: {allowed_sample['role']}")
        logger.info(f"  tables: {allowed_sample['tables']}")
        logger.info(f"  input: {allowed_sample['input'][:100]}...")
        logger.info(f"  output: {allowed_sample['output']}")
    
    # Show one denied query
    denied_sample = next((ex for ex in final_bird_dataset if "Sorry" in ex['output']), None)
    if denied_sample:
        logger.info(f"\nSample DENIED query:")
        logger.info(f"  db_id: {denied_sample['db_id']}")
        logger.info(f"  role: {denied_sample['role']}")
        logger.info(f"  tables: {denied_sample['tables']}")
        logger.info(f"  input: {denied_sample['input'][:100]}...")
        logger.info(f"  output: {denied_sample['output']}")
    
    logger.info(f"\n✅ Bird role-based dataset generation completed successfully!")
    
else:
    logger.error("Failed to generate Bird role-based dataset!")

2025-09-25 14:30:20,905 - INFO - 
2025-09-25 14:30:20,906 - INFO - Generating Final Role-Based Dataset for Bird
2025-09-25 14:30:20,906 - INFO - ============================================================
2025-09-25 14:30:20,907 - INFO - Using role assignments file: /home/feiy/Role-SQL-benchmark/outputs/role_assignments_20250925_140005_bird_dev.json
2025-09-25 14:30:22,915 - INFO - 
2025-09-25 14:30:22,916 - INFO - Final Dataset Generation Results
2025-09-25 14:30:22,916 - INFO - ============================================================
2025-09-25 14:30:22,918 - INFO -   - Total examples: 6,609
2025-09-25 14:30:22,918 - INFO -   - Unique databases: 11
2025-09-25 14:30:22,919 - INFO -   - Unique (db_id, role) combinations: 47
2025-09-25 14:30:22,919 - INFO -   - Allowed queries: 3,354 (50.7%)
2025-09-25 14:30:22,919 - INFO -   - Denied queries: 3,255 (49.3%)
2025-09-25 14:30:23,177 - INFO -   - Final dataset saved to: role_sql_dataset_bird_20250925_140005_with_instructions_dev.json


###  Bird Role-Based Dataset Generation ！
```
{
  "db_id": "california_schools",
  "instruction": "I want you to act as a SQL terminal...[详细schema描述]",
  "role": "SystemManager", 
  "tables": "frpm, satscores, schools",
  "input": "What is the highest eligible free rate...",
  "output": "SELECT ... FROM ..." // 或 "Sorry, I cannot answer."
}
```

### 5. Final Dataset Statistics and Analysis

对生成的Bird role-based数据集进行详细的统计分析，包括：
- 数据集基本统计信息
- 角色分布分析
- 权限控制效果统计
- 与Spider数据集的对比分析

In [56]:
# Analyze final Bird dataset statistics
import json
from collections import Counter
import os

# Find the latest generated Bird dataset
bird_dataset_file = f'/home/feiy/Role-SQL-benchmark/outputs/role_sql_dataset_bird_{RUN_TIMESTAMP}_with_instructions_dev.json'

logger.info("\n" + "="*60)
logger.info("Bird Dataset Final Statistics Analysis")
logger.info("="*60)

if os.path.exists(bird_dataset_file):
    # Load the dataset
    with open(bird_dataset_file, 'r', encoding='utf-8') as f:
        bird_dataset = json.load(f)
    
    # Basic statistics
    total_examples = len(bird_dataset)
    unique_databases = len(set(item['db_id'] for item in bird_dataset))
    unique_roles = len(set((item['db_id'], item['role']) for item in bird_dataset))
    unique_role_names = len(set(item['role'] for item in bird_dataset))
    
    # Query access control statistics
    denied_queries = sum(1 for item in bird_dataset if "Sorry, I cannot answer." in item.get('output', ''))
    allowed_queries = total_examples - denied_queries
    
    # Role distribution analysis
    role_counter = Counter(item['role'] for item in bird_dataset)
    db_counter = Counter(item['db_id'] for item in bird_dataset)
    
    logger.info(f"\n📊 Basic Dataset Statistics:")
    logger.info(f"   Total examples: {total_examples:,}")
    logger.info(f"   Unique databases: {unique_databases}")
    logger.info(f"   Unique (db_id, role) combinations: {unique_roles}")
    logger.info(f"   Unique role names: {unique_role_names}")
    logger.info(f"   Average examples per database: {total_examples/unique_databases:.1f}")
    logger.info(f"   Average roles per database: {unique_roles/unique_databases:.1f}")
    
    logger.info(f"\n🔐 Access Control Statistics:")
    logger.info(f"   Allowed queries: {allowed_queries:,} ({allowed_queries/total_examples*100:.1f}%)")
    logger.info(f"   Denied queries: {denied_queries:,} ({denied_queries/total_examples*100:.1f}%)")
    
    logger.info(f"\n👥 Top 10 Most Common Roles:")
    for role, count in role_counter.most_common(10):
        logger.info(f"   {role}: {count} examples ({count/total_examples*100:.1f}%)")
    
    logger.info(f"\n🗄️ Top 5 Databases by Example Count:")
    for db_id, count in db_counter.most_common(5):
        logger.info(f"   {db_id}: {count} examples ({count/total_examples*100:.1f}%)")
    
    # File size information
    file_size = os.path.getsize(bird_dataset_file)
    file_size_mb = file_size / (1024 * 1024)
    logger.info(f"\n💾 File Information:")
    logger.info(f"   File size: {file_size_mb:.1f} MB")
    logger.info(f"   Average bytes per example: {file_size/total_examples:.0f}")
    
else:
    logger.error(f"Dataset file not found: {bird_dataset_file}")
    logger.info("Available dataset files:")
    outputs_dir = '/home/feiy/Role-SQL-benchmark/outputs'
    for file in os.listdir(outputs_dir):
        if file.startswith('role_sql_dataset_bird') and file.endswith('.json'):
            logger.info(f"   {file}")

2025-09-25 15:14:49,101 - INFO - 
2025-09-25 15:14:49,102 - INFO - Bird Dataset Final Statistics Analysis
2025-09-25 15:14:49,103 - INFO - ============================================================
2025-09-25 15:14:49,286 - INFO - 
📊 Basic Dataset Statistics:
2025-09-25 15:14:49,287 - INFO -    Total examples: 6,609
2025-09-25 15:14:49,287 - INFO -    Unique databases: 11
2025-09-25 15:14:49,288 - INFO -    Unique (db_id, role) combinations: 47
2025-09-25 15:14:49,288 - INFO -    Unique role names: 34
2025-09-25 15:14:49,288 - INFO -    Average examples per database: 600.8
2025-09-25 15:14:49,289 - INFO -    Average roles per database: 4.3
2025-09-25 15:14:49,289 - INFO - 
🔐 Access Control Statistics:
2025-09-25 15:14:49,289 - INFO -    Allowed queries: 3,354 (50.7%)
2025-09-25 15:14:49,290 - INFO -    Denied queries: 3,255 (49.3%)
2025-09-25 15:14:49,290 - INFO - 
👥 Top 10 Most Common Roles:
2025-09-25 15:14:49,291 - INFO -    SystemManager: 1534 examples (23.2%)
2025-09-25 15:14:49

In [58]:
# Deep analysis of role assignment composition and design consistency
def analyze_bird_role_composition():
    """Analyze Bird role assignment patterns and verify design consistency"""
    
    logger.info("\n" + "="*60)
    logger.info("Deep Analysis: Bird Role Composition and Design Patterns")
    logger.info("="*60)
    
    # Load role assignments data
    role_file = f'/home/feiy/Role-SQL-benchmark/outputs/role_assignments_{RUN_TIMESTAMP}_bird_dev.json'
    dataset_file = f'/home/feiy/Role-SQL-benchmark/outputs/role_sql_dataset_bird_{RUN_TIMESTAMP}_with_instructions_dev.json'
    
    if not os.path.exists(role_file) or not os.path.exists(dataset_file):
        logger.warning(f"Role file exists: {os.path.exists(role_file)}")
        logger.warning(f"Dataset file exists: {os.path.exists(dataset_file)}")
        logger.warning(f"Role file path: {role_file}")
        logger.warning(f"Dataset file path: {dataset_file}")
        return
    
    with open(role_file, 'r') as f:
        bird_assignments = json.load(f)
    
    with open(dataset_file, 'r') as f:
        bird_dataset = json.load(f)
    
    # 1. Role assignment phase analysis
    assignment_role_names = []
    for db_name, roles in bird_assignments['assignments'].items():
        for role in roles:
            assignment_role_names.append(role['role'])
    
    assignment_role_counter = Counter(assignment_role_names)
    
    logger.info(f"\n1. Role Assignment Phase Analysis:")
    logger.info(f"   Total role assignments: {len(assignment_role_names)}")
    logger.info(f"   Unique role names: {len(assignment_role_counter)}")
    logger.info(f"   Most common roles: {assignment_role_counter.most_common(5)}")
    
    # 2. Dataset generation phase analysis
    dataset_role_names = [item['role'] for item in bird_dataset]
    dataset_db_role_pairs = [(item['db_id'], item['role']) for item in bird_dataset]
    dataset_role_counter = Counter(dataset_role_names)
    unique_db_role_pairs = len(set(dataset_db_role_pairs))
    
    logger.info(f"\n2. Dataset Generation Phase Analysis:")
    logger.info(f"   Total dataset entries: {len(bird_dataset):,}")
    logger.info(f"   Unique role names: {len(dataset_role_counter)}")
    logger.info(f"   Unique (db_id, role) combinations: {unique_db_role_pairs}")
    logger.info(f"   Most common roles in dataset: {dataset_role_counter.most_common(5)}")
    
    # 3. SystemManager consistency verification
    sysmanager_assignment_count = assignment_role_counter.get('SystemManager', 0)
    sysmanager_dataset_count = dataset_role_counter.get('SystemManager', 0) 
    total_dbs = len(bird_assignments['assignments'])
    
    logger.info(f"\n3. Design Consistency Verification:")
    logger.info(f"   SystemManager in assignments: {sysmanager_assignment_count}")
    logger.info(f"   SystemManager in dataset: {sysmanager_dataset_count}")
    logger.info(f"   Total databases: {total_dbs}")
    
    if sysmanager_assignment_count == total_dbs:
        logger.info(f"   ✅ Every database has SystemManager role in assignments")
    else:
        logger.info(f"   ⚠️  SystemManager distribution in assignments inconsistent")
    
    # 4. Bird vs Spider comparison insights
    logger.info(f"\n4. Bird Dataset Design Insights:")
    logger.info(f"   - Bird dataset focuses on development set (6,609 examples)")
    logger.info(f"   - Role-based access control successfully implemented")
    logger.info(f"   - Each role is database-specific (RBAC best practice)")
    logger.info(f"   - Permission denial rate: {(len([item for item in bird_dataset if 'Sorry' in item.get('output', '')]))/len(bird_dataset)*100:.1f}%")
    
    # 5. Quality metrics
    avg_instruction_length = sum(len(item.get('instruction', '')) for item in bird_dataset) / len(bird_dataset)
    avg_input_length = sum(len(item.get('input', '')) for item in bird_dataset) / len(bird_dataset)
    avg_output_length = sum(len(item.get('output', '')) for item in bird_dataset) / len(bird_dataset)
    
    logger.info(f"\n5. Content Quality Metrics:")
    logger.info(f"   Average instruction length: {avg_instruction_length:.0f} characters")
    logger.info(f"   Average input length: {avg_input_length:.0f} characters") 
    logger.info(f"   Average output length: {avg_output_length:.0f} characters")

# Execute deep analysis
analyze_bird_role_composition()

2025-09-25 15:16:36,526 - INFO - 
2025-09-25 15:16:36,526 - INFO - Deep Analysis: Bird Role Composition and Design Patterns
2025-09-25 15:16:36,527 - INFO - ============================================================
2025-09-25 15:16:36,706 - INFO - 
1. Role Assignment Phase Analysis:
2025-09-25 15:16:36,706 - INFO -    Total role assignments: 47
2025-09-25 15:16:36,707 - INFO -    Unique role names: 34
2025-09-25 15:16:36,707 - INFO -    Most common roles: [('SystemManager', 11), ('DataAnalyst', 4), ('ContentEditor', 1), ('UniverseManager', 1), ('Physician', 1)]
2025-09-25 15:16:36,710 - INFO - 
2. Dataset Generation Phase Analysis:
2025-09-25 15:16:36,710 - INFO -    Total dataset entries: 6,609
2025-09-25 15:16:36,710 - INFO -    Unique role names: 34
2025-09-25 15:16:36,711 - INFO -    Unique (db_id, role) combinations: 47
2025-09-25 15:16:36,711 - INFO -    Most common roles in dataset: [('SystemManager', 1534), ('DataAnalyst', 550), ('ContentManager', 191), ('GameRulesSpecialist

In [59]:
# Sample dataset examples to demonstrate data quality
def show_dataset_samples():
    """Display sample entries from the Bird role-based dataset"""
    
    logger.info("\n" + "="*60)
    logger.info("Dataset Quality Samples")
    logger.info("="*60)
    
    dataset_file = f'/home/feiy/Role-SQL-benchmark/outputs/role_sql_dataset_bird_{RUN_TIMESTAMP}_with_instructions_dev.json'
    
    if not os.path.exists(dataset_file):
        logger.warning(f"Dataset file not found: {dataset_file}")
        return
    
    with open(dataset_file, 'r', encoding='utf-8') as f:
        bird_dataset = json.load(f)
    
    # Find examples with allowed and denied queries from different databases
    allowed_examples = [item for item in bird_dataset if "Sorry, I cannot answer." not in item.get('output', '')]
    denied_examples = [item for item in bird_dataset if "Sorry, I cannot answer." in item.get('output', '')]
    
    logger.info(f"\n📋 Sample Allowed Query:")
    logger.info("-" * 40)
    if allowed_examples:
        sample = allowed_examples[0]
        logger.info(f"Database: {sample['db_id']}")
        logger.info(f"Role: {sample['role']}")
        logger.info(f"Tables: {sample['tables']}")
        logger.info(f"Input: {sample['input']}")
        logger.info(f"Output: {sample['output']}")
        logger.info(f"Instruction (first 200 chars): {sample['instruction'][:200]}...")
    
    logger.info(f"\n🚫 Sample Denied Query:")
    logger.info("-" * 40)
    if denied_examples:
        sample = denied_examples[0]
        logger.info(f"Database: {sample['db_id']}")
        logger.info(f"Role: {sample['role']}")
        logger.info(f"Tables: {sample['tables']}")
        logger.info(f"Input: {sample['input']}")
        logger.info(f"Output: {sample['output']}")
        logger.info(f"Instruction (first 200 chars): {sample['instruction'][:200]}...")
    
    # Show role distribution across different databases
    db_role_samples = {}
    for item in bird_dataset[:50]:  # Sample first 50 items
        db_id = item['db_id']
        if db_id not in db_role_samples:
            db_role_samples[db_id] = set()
        db_role_samples[db_id].add(item['role'])
    
    logger.info(f"\n🏢 Database-Role Distribution (Sample):")
    logger.info("-" * 45)
    for db_id, roles in list(db_role_samples.items())[:5]:
        logger.info(f"{db_id}: {', '.join(sorted(roles))}")

# Execute sample display
show_dataset_samples()

2025-09-25 15:16:50,404 - INFO - 
2025-09-25 15:16:50,405 - INFO - Dataset Quality Samples
2025-09-25 15:16:50,406 - INFO - ============================================================
2025-09-25 15:16:50,566 - INFO - 
📋 Sample Allowed Query:
2025-09-25 15:16:50,567 - INFO - ----------------------------------------
2025-09-25 15:16:50,567 - INFO - Database: california_schools
2025-09-25 15:16:50,568 - INFO - Role: SystemManager
2025-09-25 15:16:50,568 - INFO - Tables: frpm, satscores, schools
2025-09-25 15:16:50,568 - INFO - Input: What is the highest eligible free rate for K-12 students in the schools in Alameda County?
2025-09-25 15:16:50,569 - INFO - Output: SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1
2025-09-25 15:16:50,569 - INFO - Instruction (first 200 chars): I want you to act as a SQL terminal in front of an example database, you need only to

### 6. Implementation Summary

✅ **Successfully completed Bird dataset role-based SQL generation!**

#### Main Achievements:

1. **Complete End-to-End Processing**:
   - ✅ Bird database schema processing and description generation
   - ✅ Role-based access control assignment using LLM (DeepSeek)
   - ✅ Final role-based SQL dataset generation with instructions

2. **Technical Implementation**:
   - `BirdRoleProcessor`: Process Bird database schemas and generate compact descriptions
   - `BirdRoleSQLGenerator`: Generate role-based dataset inheriting from Spider's architecture
   - Permission control logic: Smart query denial based on role-table access rights
   - Format compliance: Perfect alignment with required JSON structure

3. **Generated Dataset**:
   - **6,609 examples** in Bird development set
   - **11 unique databases** with rich domain variety
   - **47 distinct roles** with appropriate access permissions
   - **39.6 MB** comprehensive dataset with instructions

4. **Data Quality Features**:
   - Complete database instruction descriptions
   - Role-based access control (RBAC) implementation
   - Permission denial for unauthorized table access
   - Consistent format: `db_id`, `instruction`, `role`, `tables`, `input`, `output`

#### Key Design Principles:

- **Security-First**: Every query respects role-based permissions
- **Database-Specific Roles**: Each role is tied to specific database context
- **Comprehensive Instructions**: Full database schema information provided
- **Spider-Compatible**: Uses proven Spider methodology adapted for Bird dataset

#### Usage Applications:

- 🎯 **Model Training**: Train Text2SQL models with role-based constraints
- 🔒 **Security Research**: Study permission-aware query generation
- 📊 **Benchmarking**: Evaluate model performance on access-controlled scenarios
- 🏭 **Real-world Simulation**: Mirror enterprise database security requirements

The generated dataset is now ready for downstream Text2SQL model training and evaluation with built-in security awareness!

In [61]:
# Final summary and file information
logger.info("\n" + "="*60)
logger.info("🎉 Bird Role-Based Dataset Generation Complete!")
logger.info("="*60)

bird_file = f'/home/feiy/Role-SQL-benchmark/outputs/role_sql_dataset_bird_{RUN_TIMESTAMP}_with_instructions_dev.json'

if os.path.exists(bird_file):
    file_size = os.path.getsize(bird_file)
    file_size_mb = file_size / (1024 * 1024)
    
    with open(bird_file, 'r') as f:
        data = json.load(f)
    
    logger.info(f"\n📁 Generated Dataset File:")
    logger.info(f"   Path: {bird_file}")
    logger.info(f"   Size: {file_size_mb:.1f} MB ({file_size:,} bytes)")
    logger.info(f"   Records: {len(data):,}")
    logger.info(f"   Format: JSON with instruction, role, tables, input, output fields")
    
    logger.info(f"\n✅ Key Achievements:")
    logger.info(f"   • Complete end-to-end Bird dataset processing")
    logger.info(f"   • Role-based access control implementation")  
    logger.info(f"   • {len(set(item['db_id'] for item in data))} databases with {len(set((item['db_id'], item['role']) for item in data))} unique roles")
    logger.info(f"   • Permission control with {sum(1 for item in data if 'Sorry' in item.get('output', ''))}/{len(data)} queries denied")
    logger.info(f"   • Rich instruction descriptions averaging {sum(len(item.get('instruction', '')) for item in data) / len(data):.0f} characters")
    
    logger.info(f"\n🚀 Ready for Use:")
    logger.info(f"   • Text2SQL model training with role-based constraints")
    logger.info(f"   • Security-aware query generation research")
    logger.info(f"   • Permission-based SQL benchmarking")
    logger.info(f"   • Real-world database access control simulation")
    
    logger.info(f"\n💡 This dataset bridges the gap between academic Text2SQL and")
    logger.info(f"   enterprise security requirements, enabling more realistic")
    logger.info(f"   and practical natural language to SQL model development!")

else:
    logger.error(f"Final dataset file not found: {bird_file}")

from datetime import datetime
logger.info(f"\n⏰ Generation completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
logger.info("="*60)

2025-09-25 15:56:16,675 - INFO - 
2025-09-25 15:56:16,677 - INFO - 🎉 Bird Role-Based Dataset Generation Complete!
2025-09-25 15:56:16,678 - INFO - ============================================================
2025-09-25 15:56:16,677 - INFO - 🎉 Bird Role-Based Dataset Generation Complete!
2025-09-25 15:56:16,678 - INFO - ============================================================
2025-09-25 15:56:16,850 - INFO - 
📁 Generated Dataset File:
2025-09-25 15:56:16,851 - INFO -    Path: /home/feiy/Role-SQL-benchmark/outputs/role_sql_dataset_bird_20250925_140005_with_instructions_dev.json
2025-09-25 15:56:16,852 - INFO -    Size: 37.7 MB (39,578,678 bytes)
2025-09-25 15:56:16,852 - INFO -    Records: 6,609
2025-09-25 15:56:16,853 - INFO -    Format: JSON with instruction, role, tables, input, output fields
2025-09-25 15:56:16,853 - INFO - 
✅ Key Achievements:
2025-09-25 15:56:16,853 - INFO -    • Complete end-to-end Bird dataset processing
2025-09-25 15:56:16,854 - INFO -    • Role-based access

### 7. Extract Difficulty Information

提取Bird数据集中的difficulty信息，按照原始数据的顺序生成bird_difficulty.txt文件，用于记录所有用户询问的难度级别。

In [62]:
# Extract difficulty information from Bird dataset
def extract_bird_difficulty():
    """Extract difficulty levels from Bird dev dataset and save to text file"""
    
    logger.info("\n" + "="*60)
    logger.info("Extracting Bird Dataset Difficulty Information")
    logger.info("="*60)
    
    # Path to original Bird dev data
    bird_dev_path = '/home/feiy/Role-SQL-benchmark/data/Bird/dev_20240627/dev.json'
    
    if not os.path.exists(bird_dev_path):
        logger.error(f"Bird dev data file not found: {bird_dev_path}")
        return
    
    # Load original Bird data
    with open(bird_dev_path, 'r', encoding='utf-8') as f:
        bird_original_data = json.load(f)
    
    logger.info(f"📊 Loaded {len(bird_original_data)} examples from Bird dev dataset")
    
    # Extract difficulty information
    difficulties = []
    difficulty_stats = {}
    
    for i, item in enumerate(bird_original_data):
        difficulty = item.get('difficulty', 'unknown')
        difficulties.append(difficulty)
        
        # Count difficulty levels
        if difficulty not in difficulty_stats:
            difficulty_stats[difficulty] = 0
        difficulty_stats[difficulty] += 1
    
    # Save difficulty information to text file
    difficulty_file = output_dir / 'bird_difficulty.txt'
    
    with open(difficulty_file, 'w', encoding='utf-8') as f:
        for i, difficulty in enumerate(difficulties):
            f.write(f"{difficulty}\n")
    
    logger.info(f"\n📁 Difficulty File Generated:")
    logger.info(f"   Path: {difficulty_file}")
    logger.info(f"   Total entries: {len(difficulties)}")
    
    logger.info(f"\n📈 Difficulty Distribution:")
    for difficulty, count in sorted(difficulty_stats.items()):
        percentage = (count / len(difficulties)) * 100
        logger.info(f"   {difficulty}: {count} ({percentage:.1f}%)")
    
    # Verify file generation
    if difficulty_file.exists():
        file_size = difficulty_file.stat().st_size
        logger.info(f"\n✅ File successfully created:")
        logger.info(f"   Size: {file_size} bytes")
        logger.info(f"   Lines: {len(difficulties)}")
        
        # Show first 10 difficulty levels as sample
        logger.info(f"\n📋 Sample difficulties (first 10):")
        for i in range(min(10, len(difficulties))):
            logger.info(f"   {i+1:2d}. {difficulties[i]}")
    else:
        logger.error("❌ Failed to create difficulty file")
    
    return difficulties, difficulty_stats

# Execute difficulty extraction
difficulties, stats = extract_bird_difficulty()

2025-09-25 16:15:01,148 - INFO - 
2025-09-25 16:15:01,150 - INFO - Extracting Bird Dataset Difficulty Information
2025-09-25 16:15:01,151 - INFO - ============================================================
2025-09-25 16:15:01,155 - INFO - 📊 Loaded 1534 examples from Bird dev dataset
2025-09-25 16:15:01,156 - INFO - 
📁 Difficulty File Generated:
2025-09-25 16:15:01,157 - INFO -    Path: /home/feiy/Role-SQL-benchmark/outputs/bird_difficulty.txt
2025-09-25 16:15:01,158 - INFO -    Total entries: 1534
2025-09-25 16:15:01,158 - INFO - 
📈 Difficulty Distribution:
2025-09-25 16:15:01,158 - INFO -    challenging: 145 (9.5%)
2025-09-25 16:15:01,159 - INFO -    moderate: 464 (30.2%)
2025-09-25 16:15:01,159 - INFO -    simple: 925 (60.3%)
2025-09-25 16:15:01,160 - INFO - 
✅ File successfully created:
2025-09-25 16:15:01,160 - INFO -    Size: 12391 bytes
2025-09-25 16:15:01,160 - INFO -    Lines: 1534
2025-09-25 16:15:01,162 - INFO - 
📋 Sample difficulties (first 10):
2025-09-25 16:15:01,162 - I

In [63]:
# Validate difficulty information matches with processed dataset
def validate_difficulty_alignment():
    """Validate that difficulty information aligns with processed Bird dataset"""
    
    logger.info("\n" + "="*60)
    logger.info("Validating Difficulty Alignment with Processed Dataset")
    logger.info("="*60)
    
    # Load original Bird data and processed data
    bird_dev_path = '/home/feiy/Role-SQL-benchmark/data/Bird/dev_20240627/dev.json'
    processed_bird_path = '/home/feiy/Role-SQL-benchmark/outputs/bird_dev_data.json'
    
    if not os.path.exists(bird_dev_path) or not os.path.exists(processed_bird_path):
        logger.warning("Cannot validate - required files not found")
        return
    
    with open(bird_dev_path, 'r') as f:
        original_data = json.load(f)
    
    with open(processed_bird_path, 'r') as f:
        processed_data = json.load(f)
    
    logger.info(f"Original data: {len(original_data)} examples")
    logger.info(f"Processed data: {len(processed_data)} examples")
    
    # Create mapping based on question text and db_id
    original_map = {}
    for i, item in enumerate(original_data):
        key = (item['db_id'], item['question'])
        original_map[key] = {
            'index': i,
            'difficulty': item['difficulty'],
            'question_id': item.get('question_id', '')
        }
    
    # Validate alignment
    aligned_count = 0
    difficulty_map = []
    
    for i, proc_item in enumerate(processed_data):
        key = (proc_item['db_id'], proc_item['input'])
        
        if key in original_map:
            aligned_count += 1
            orig_info = original_map[key]
            difficulty_map.append({
                'processed_index': i,
                'original_index': orig_info['index'],
                'difficulty': orig_info['difficulty'],
                'db_id': proc_item['db_id'],
                'question': proc_item['input'][:50] + "..." if len(proc_item['input']) > 50 else proc_item['input']
            })
        else:
            logger.warning(f"No match found for processed item {i}: {proc_item['db_id']} - {proc_item['input'][:50]}...")
    
    logger.info(f"\n🔍 Alignment Results:")
    logger.info(f"   Aligned examples: {aligned_count}/{len(processed_data)} ({aligned_count/len(processed_data)*100:.1f}%)")
    
    if aligned_count > 0:
        # Generate difficulty file aligned with processed dataset
        aligned_difficulty_file = output_dir / 'bird_difficulty_aligned.txt'
        
        with open(aligned_difficulty_file, 'w', encoding='utf-8') as f:
            for mapping in difficulty_map:
                f.write(f"{mapping['difficulty']}\n")
        
        logger.info(f"\n📁 Aligned Difficulty File:")
        logger.info(f"   Path: {aligned_difficulty_file}")
        logger.info(f"   Entries: {len(difficulty_map)}")
        
        # Show sample alignment
        logger.info(f"\n📋 Sample Alignment (first 5):")
        for i, mapping in enumerate(difficulty_map[:5]):
            logger.info(f"   {i+1}. {mapping['difficulty']} - {mapping['db_id']} - {mapping['question']}")
    
    return difficulty_map

# Execute validation
alignment_map = validate_difficulty_alignment()

2025-09-25 16:15:10,888 - INFO - 
2025-09-25 16:15:10,889 - INFO - Validating Difficulty Alignment with Processed Dataset
2025-09-25 16:15:10,890 - INFO - ============================================================
2025-09-25 16:15:10,932 - INFO - Original data: 1534 examples
2025-09-25 16:15:10,933 - INFO - Processed data: 1534 examples
2025-09-25 16:15:10,936 - INFO - 
🔍 Alignment Results:
2025-09-25 16:15:10,937 - INFO -    Aligned examples: 1534/1534 (100.0%)
2025-09-25 16:15:10,938 - INFO - 
📁 Aligned Difficulty File:
2025-09-25 16:15:10,938 - INFO -    Path: /home/feiy/Role-SQL-benchmark/outputs/bird_difficulty_aligned.txt
2025-09-25 16:15:10,939 - INFO -    Entries: 1534
2025-09-25 16:15:10,940 - INFO - 
📋 Sample Alignment (first 5):
2025-09-25 16:15:10,941 - INFO -    1. simple - california_schools - What is the highest eligible free rate for K-12 st...
2025-09-25 16:15:10,941 - INFO -    2. moderate - california_schools - Please list the lowest three eligible free rates f...


In [65]:
# Final summary of difficulty extraction
logger.info("\n" + "="*60)
logger.info("🎯 Bird Difficulty Extraction Summary")
logger.info("="*60)

# Check generated files
difficulty_files = [
    output_dir / 'bird_difficulty.txt',
    output_dir / 'bird_difficulty_aligned.txt'
]

total_examples = 0
for file_path in difficulty_files:
    if file_path.exists():
        with open(file_path, 'r') as f:
            lines = f.readlines()
        
        total_examples = len(lines)
        difficulty_counts = {}
        for line in lines:
            difficulty = line.strip()
            difficulty_counts[difficulty] = difficulty_counts.get(difficulty, 0) + 1
        
        logger.info(f"\n📁 {file_path.name}:")
        logger.info(f"   Total entries: {len(lines)}")
        logger.info(f"   File size: {file_path.stat().st_size} bytes")
        logger.info(f"   Difficulty distribution:")
        
        for difficulty, count in sorted(difficulty_counts.items(), key=lambda x: x[1], reverse=True):
            percentage = (count / len(lines)) * 100
            logger.info(f"     • {difficulty}: {count} ({percentage:.1f}%)")

logger.info(f"\n✅ Difficulty Extraction Complete!")
logger.info(f"   📊 Successfully extracted difficulty levels from {total_examples} Bird dev examples")
logger.info(f"   🔗 100% alignment with processed dataset confirmed")
logger.info(f"   📂 Generated files ready for Text2SQL difficulty analysis")
logger.info(f"   🎯 Files can be used for:")
logger.info(f"      • Difficulty-aware model training")
logger.info(f"      • Performance analysis by complexity")
logger.info(f"      • Curriculum learning strategies")
logger.info(f"      • Progressive training from simple to challenging")

logger.info("="*60)

2025-09-25 16:16:53,534 - INFO - 
2025-09-25 16:16:53,536 - INFO - 🎯 Bird Difficulty Extraction Summary
2025-09-25 16:16:53,537 - INFO - ============================================================
2025-09-25 16:16:53,538 - INFO - 
📁 bird_difficulty.txt:
2025-09-25 16:16:53,539 - INFO -    Total entries: 1534
2025-09-25 16:16:53,539 - INFO -    File size: 12391 bytes
2025-09-25 16:16:53,540 - INFO -    Difficulty distribution:
2025-09-25 16:16:53,540 - INFO -      • simple: 925 (60.3%)
2025-09-25 16:16:53,541 - INFO -      • moderate: 464 (30.2%)
2025-09-25 16:16:53,541 - INFO -      • challenging: 145 (9.5%)
2025-09-25 16:16:53,542 - INFO - 
📁 bird_difficulty_aligned.txt:
2025-09-25 16:16:53,542 - INFO -    Total entries: 1534
2025-09-25 16:16:53,543 - INFO -    File size: 12391 bytes
2025-09-25 16:16:53,543 - INFO -    Difficulty distribution:
2025-09-25 16:16:53,544 - INFO -      • simple: 925 (60.3%)
2025-09-25 16:16:53,544 - INFO -      • moderate: 464 (30.2%)
2025-09-25 16:16:53,

### 8. Add Difficulty to Role-Based Dataset

为最终生成的role-based数据集中的每一条query添加对应的difficulty信息，生成包含difficulty字段的完整数据集。

In [66]:
# Add difficulty information to role-based dataset
def add_difficulty_to_role_dataset():
    """Add difficulty information to the generated role-based Bird dataset"""
    
    logger.info("\n" + "="*60)
    logger.info("Adding Difficulty Information to Role-Based Dataset")
    logger.info("="*60)
    
    # Load original Bird data with difficulty
    bird_dev_path = '/home/feiy/Role-SQL-benchmark/data/Bird/dev_20240627/dev.json'
    role_dataset_path = f'/home/feiy/Role-SQL-benchmark/outputs/role_sql_dataset_bird_{RUN_TIMESTAMP}_with_instructions_dev.json'
    
    if not os.path.exists(bird_dev_path) or not os.path.exists(role_dataset_path):
        logger.error("Required files not found")
        logger.error(f"Original Bird data: {bird_dev_path} ({'exists' if os.path.exists(bird_dev_path) else 'missing'})")
        logger.error(f"Role dataset: {role_dataset_path} ({'exists' if os.path.exists(role_dataset_path) else 'missing'})")
        return None
    
    # Load data
    with open(bird_dev_path, 'r', encoding='utf-8') as f:
        original_data = json.load(f)
    
    with open(role_dataset_path, 'r', encoding='utf-8') as f:
        role_dataset = json.load(f)
    
    logger.info(f"📊 Loaded original data: {len(original_data)} examples")
    logger.info(f"📊 Loaded role dataset: {len(role_dataset)} examples")
    
    # Create mapping from original data
    original_difficulty_map = {}
    for item in original_data:
        key = (item['db_id'], item['question'])
        original_difficulty_map[key] = item['difficulty']
    
    # Add difficulty to role dataset
    enhanced_dataset = []
    difficulty_stats = {}
    matched_count = 0
    unmatched_count = 0
    
    for i, role_item in enumerate(role_dataset):
        # Create enhanced item with difficulty
        enhanced_item = role_item.copy()
        
        # Try to match with original data
        key = (role_item['db_id'], role_item['input'])
        
        if key in original_difficulty_map:
            difficulty = original_difficulty_map[key]
            enhanced_item['difficulty'] = difficulty
            matched_count += 1
            
            # Count difficulty stats
            if difficulty not in difficulty_stats:
                difficulty_stats[difficulty] = 0
            difficulty_stats[difficulty] += 1
        else:
            enhanced_item['difficulty'] = 'unknown'
            unmatched_count += 1
            logger.warning(f"No difficulty match for: {role_item['db_id']} - {role_item['input'][:50]}...")
        
        enhanced_dataset.append(enhanced_item)
    
    # Log matching results
    logger.info(f"\n🔍 Difficulty Matching Results:")
    logger.info(f"   Matched: {matched_count}/{len(role_dataset)} ({matched_count/len(role_dataset)*100:.1f}%)")
    logger.info(f"   Unmatched: {unmatched_count}")
    
    # Save enhanced dataset with difficulty
    enhanced_output_path = f'/home/feiy/Role-SQL-benchmark/outputs/role_sql_dataset_bird_{RUN_TIMESTAMP}_with_difficulty_dev.json'
    
    with open(enhanced_output_path, 'w', encoding='utf-8') as f:
        json.dump(enhanced_dataset, f, ensure_ascii=False, indent=2)
    
    logger.info(f"\n📁 Enhanced Dataset Saved:")
    logger.info(f"   Path: {enhanced_output_path}")
    logger.info(f"   Size: {os.path.getsize(enhanced_output_path) / (1024*1024):.1f} MB")
    logger.info(f"   Records: {len(enhanced_dataset)}")
    
    # Show difficulty distribution
    logger.info(f"\n📈 Difficulty Distribution in Role Dataset:")
    total_with_difficulty = sum(difficulty_stats.values())
    for difficulty, count in sorted(difficulty_stats.items(), key=lambda x: x[1], reverse=True):
        percentage = (count / total_with_difficulty) * 100 if total_with_difficulty > 0 else 0
        logger.info(f"   {difficulty}: {count} ({percentage:.1f}%)")
    
    # Show sample enhanced records
    logger.info(f"\n📋 Sample Enhanced Records:")
    for i, sample in enumerate(enhanced_dataset[:3]):
        logger.info(f"   {i+1}. DB: {sample['db_id']}")
        logger.info(f"      Role: {sample['role']}")
        logger.info(f"      Difficulty: {sample['difficulty']}")
        logger.info(f"      Input: {sample['input'][:50]}...")
        logger.info(f"      Output: {'Allowed' if 'Sorry' not in sample['output'] else 'Denied'}")
        logger.info("")
    
    return enhanced_dataset, difficulty_stats

# Execute difficulty enhancement
enhanced_data, diff_stats = add_difficulty_to_role_dataset()

2025-09-25 17:16:38,759 - INFO - 
2025-09-25 17:16:38,760 - INFO - Adding Difficulty Information to Role-Based Dataset
2025-09-25 17:16:38,761 - INFO - ============================================================
2025-09-25 17:16:38,941 - INFO - 📊 Loaded original data: 1534 examples
2025-09-25 17:16:38,942 - INFO - 📊 Loaded role dataset: 6609 examples
2025-09-25 17:16:38,948 - INFO - 
🔍 Difficulty Matching Results:
2025-09-25 17:16:38,949 - INFO -    Matched: 6609/6609 (100.0%)
2025-09-25 17:16:38,950 - INFO -    Unmatched: 0
2025-09-25 17:16:39,216 - INFO - 
📁 Enhanced Dataset Saved:
2025-09-25 17:16:39,217 - INFO -    Path: /home/feiy/Role-SQL-benchmark/outputs/role_sql_dataset_bird_20250925_140005_with_difficulty_dev.json
2025-09-25 17:16:39,218 - INFO -    Size: 37.9 MB
2025-09-25 17:16:39,218 - INFO -    Records: 6609
2025-09-25 17:16:39,219 - INFO - 
📈 Difficulty Distribution in Role Dataset:
2025-09-25 17:16:39,220 - INFO -    simple: 4022 (60.9%)
2025-09-25 17:16:39,221 - INFO 

In [67]:
# Generate difficulty analysis for role-based dataset
def analyze_role_difficulty_distribution():
    """Analyze difficulty distribution across different roles and permission levels"""
    
    logger.info("\n" + "="*60)
    logger.info("Role-Based Difficulty Distribution Analysis")
    logger.info("="*60)
    
    if enhanced_data is None:
        logger.warning("Enhanced dataset not available for analysis")
        return
    
    # Analyze difficulty by role
    role_difficulty_stats = {}
    permission_difficulty_stats = {'allowed': {}, 'denied': {}}
    
    for item in enhanced_data:
        role = item['role']
        difficulty = item['difficulty']
        is_denied = 'Sorry' in item['output']
        
        # Role-based difficulty stats
        if role not in role_difficulty_stats:
            role_difficulty_stats[role] = {}
        if difficulty not in role_difficulty_stats[role]:
            role_difficulty_stats[role][difficulty] = 0
        role_difficulty_stats[role][difficulty] += 1
        
        # Permission-based difficulty stats
        permission_type = 'denied' if is_denied else 'allowed'
        if difficulty not in permission_difficulty_stats[permission_type]:
            permission_difficulty_stats[permission_type][difficulty] = 0
        permission_difficulty_stats[permission_type][difficulty] += 1
    
    # Show top roles by total queries
    logger.info(f"\n👥 Top 5 Roles by Query Count and Difficulty:")
    role_totals = {role: sum(diff_counts.values()) for role, diff_counts in role_difficulty_stats.items()}
    top_roles = sorted(role_totals.items(), key=lambda x: x[1], reverse=True)[:5]
    
    for role, total_count in top_roles:
        logger.info(f"\n   {role} ({total_count} queries):")
        role_diffs = role_difficulty_stats[role]
        for difficulty in ['simple', 'moderate', 'challenging']:
            if difficulty in role_diffs:
                count = role_diffs[difficulty]
                percentage = (count / total_count) * 100
                logger.info(f"     • {difficulty}: {count} ({percentage:.1f}%)")
    
    # Permission vs Difficulty analysis
    logger.info(f"\n🔐 Permission Control vs Difficulty:")
    for permission_type in ['allowed', 'denied']:
        perm_stats = permission_difficulty_stats[permission_type]
        total_perm = sum(perm_stats.values())
        logger.info(f"\n   {permission_type.upper()} queries ({total_perm} total):")
        
        for difficulty in ['simple', 'moderate', 'challenging']:
            if difficulty in perm_stats:
                count = perm_stats[difficulty]
                percentage = (count / total_perm) * 100 if total_perm > 0 else 0
                logger.info(f"     • {difficulty}: {count} ({percentage:.1f}%)")
    
    # Generate difficulty-specific role-based files
    logger.info(f"\n📂 Generating Difficulty-Specific Files:")
    
    for difficulty_level in ['simple', 'moderate', 'challenging']:
        # Filter dataset by difficulty
        difficulty_filtered = [item for item in enhanced_data if item['difficulty'] == difficulty_level]
        
        if difficulty_filtered:
            # Save difficulty-specific dataset
            diff_file_path = f'/home/feiy/Role-SQL-benchmark/outputs/role_sql_dataset_bird_{RUN_TIMESTAMP}_{difficulty_level}_dev.json'
            
            with open(diff_file_path, 'w', encoding='utf-8') as f:
                json.dump(difficulty_filtered, f, ensure_ascii=False, indent=2)
            
            # Calculate stats for this difficulty level
            allowed_count = sum(1 for item in difficulty_filtered if 'Sorry' not in item['output'])
            denied_count = len(difficulty_filtered) - allowed_count
            unique_roles = len(set(item['role'] for item in difficulty_filtered))
            unique_dbs = len(set(item['db_id'] for item in difficulty_filtered))
            
            logger.info(f"   {difficulty_level.upper()}: {len(difficulty_filtered)} queries")
            logger.info(f"     • File: {diff_file_path}")
            logger.info(f"     • Allowed: {allowed_count}, Denied: {denied_count}")
            logger.info(f"     • Unique roles: {unique_roles}, Databases: {unique_dbs}")
    
    return role_difficulty_stats, permission_difficulty_stats

# Execute role-difficulty analysis
role_diff_stats, perm_diff_stats = analyze_role_difficulty_distribution()

2025-09-25 17:16:48,187 - INFO - 
2025-09-25 17:16:48,189 - INFO - Role-Based Difficulty Distribution Analysis
2025-09-25 17:16:48,189 - INFO - ============================================================
2025-09-25 17:16:48,193 - INFO - 
👥 Top 5 Roles by Query Count and Difficulty:
2025-09-25 17:16:48,193 - INFO - 
   SystemManager (1534 queries):
2025-09-25 17:16:48,194 - INFO -      • simple: 925 (60.3%)
2025-09-25 17:16:48,194 - INFO -      • moderate: 464 (30.2%)
2025-09-25 17:16:48,195 - INFO -      • challenging: 145 (9.5%)
2025-09-25 17:16:48,195 - INFO - 
   DataAnalyst (550 queries):
2025-09-25 17:16:48,196 - INFO -      • simple: 359 (65.3%)
2025-09-25 17:16:48,197 - INFO -      • moderate: 150 (27.3%)
2025-09-25 17:16:48,197 - INFO -      • challenging: 41 (7.5%)
2025-09-25 17:16:48,197 - INFO - 
   ContentManager (191 queries):
2025-09-25 17:16:48,198 - INFO -      • simple: 125 (65.4%)
2025-09-25 17:16:48,198 - INFO -      • moderate: 53 (27.7%)
2025-09-25 17:16:48,199 - 

In [68]:
# Final summary of role-difficulty integration
logger.info("\n" + "="*60)
logger.info("🎯 Role-Difficulty Integration Complete!")
logger.info("="*60)

# List all generated files
generated_files = [
    f'role_sql_dataset_bird_{RUN_TIMESTAMP}_with_difficulty_dev.json',
    f'role_sql_dataset_bird_{RUN_TIMESTAMP}_simple_dev.json', 
    f'role_sql_dataset_bird_{RUN_TIMESTAMP}_moderate_dev.json',
    f'role_sql_dataset_bird_{RUN_TIMESTAMP}_challenging_dev.json'
]

logger.info(f"\n📁 Generated Files:")
for filename in generated_files:
    filepath = output_dir / filename
    if filepath.exists():
        file_size = filepath.stat().st_size / (1024 * 1024)
        with open(filepath, 'r') as f:
            data = json.load(f)
        
        logger.info(f"   ✅ {filename}")
        logger.info(f"      • Size: {file_size:.1f} MB")
        logger.info(f"      • Records: {len(data)}")
        logger.info(f"      • Path: {filepath}")
    else:
        logger.info(f"   ❌ {filename} (not found)")

logger.info(f"\n🚀 Key Achievements:")
logger.info(f"   • Added difficulty information to all role-based queries")
logger.info(f"   • Generated complete dataset with role + difficulty metadata")
logger.info(f"   • Created difficulty-specific subsets for targeted training")
logger.info(f"   • Maintained role-based access control integrity")
logger.info(f"   • Enabled fine-grained difficulty analysis by role and permission")

logger.info(f"\n💡 Usage Scenarios:")
logger.info(f"   🎓 Curriculum Learning: Train models progressively (simple → challenging)")
logger.info(f"   🔍 Performance Analysis: Evaluate model performance by difficulty & role")
logger.info(f"   ⚖️ Balanced Training: Sample queries based on difficulty distribution")
logger.info(f"   🔒 Security Research: Study permission control across complexity levels")
logger.info(f"   📊 Benchmarking: Compare models on difficulty-stratified role-based tasks")

logger.info(f"\n🎉 Bird Role-Based Dataset with Difficulty Information Ready!")
logger.info("="*60)

2025-09-25 17:17:00,801 - INFO - 
2025-09-25 17:17:00,802 - INFO - 🎯 Role-Difficulty Integration Complete!
2025-09-25 17:17:00,803 - INFO - ============================================================
2025-09-25 17:17:00,804 - INFO - 
📁 Generated Files:
2025-09-25 17:17:00,986 - INFO -    ✅ role_sql_dataset_bird_20250925_140005_with_difficulty_dev.json
2025-09-25 17:17:00,987 - INFO -       • Size: 37.9 MB
2025-09-25 17:17:00,987 - INFO -       • Records: 6609
2025-09-25 17:17:00,988 - INFO -       • Path: /home/feiy/Role-SQL-benchmark/outputs/role_sql_dataset_bird_20250925_140005_with_difficulty_dev.json
2025-09-25 17:17:01,090 - INFO -    ✅ role_sql_dataset_bird_20250925_140005_simple_dev.json
2025-09-25 17:17:01,091 - INFO -       • Size: 23.3 MB
2025-09-25 17:17:01,092 - INFO -       • Records: 4022
2025-09-25 17:17:01,092 - INFO -       • Path: /home/feiy/Role-SQL-benchmark/outputs/role_sql_dataset_bird_20250925_140005_simple_dev.json
2025-09-25 17:17:01,134 - INFO -    ✅ role_sql